# DNTC full OCR pipeline - Kaggle PaddleOCR/NCCL fixed version

Use Kaggle **Add-ons -> Install Dependencies** and paste install commands, not plain `requirements.txt` lines.

CPU/stable dependency block:

```bash
pip install --no-cache-dir paddlepaddle==3.2.0
pip install --no-cache-dir paddleocr==3.3.3
pip install --no-cache-dir langchain==0.3.27 langchain-community==0.3.27 langchain-text-splitters==0.3.11
pip install --no-cache-dir pymupdf pillow tqdm openpyxl gdown
```

GPU optional dependency block, only after enabling Kaggle GPU accelerator:

```bash
pip install --no-cache-dir paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
pip install --no-cache-dir paddleocr==3.3.3
pip install --no-cache-dir langchain==0.3.27 langchain-community==0.3.27 langchain-text-splitters==0.3.11
pip install --no-cache-dir pymupdf pillow tqdm openpyxl gdown
```

Run order:

```text
Add-ons -> Install Dependencies -> Save -> Run dependency install
Run -> Factory reset
Settings -> Internet -> On, if you want PaddleOCR to download models
Run All
Save Version
```

This notebook includes a fix for the Kaggle/PaddleOCR import failure:

```text
ImportError: libtorch_cuda.so: undefined symbol: ncclCommShrink
```

That error comes from PaddleOCR/PaddleX importing ModelScope, then ModelScope trying to import Kaggle's broken/incompatible Torch CUDA package. The OCR pipeline does not need Torch, so the notebook hides Torch only during the PaddleOCR import.

In [ ]:
# ============================================================
# 0. Runtime flags for PaddleOCR on Kaggle
# Must run before importing paddle / paddleocr
# ============================================================

import os

# Keep these flags before any Paddle/PaddleOCR import.
# They avoid oneDNN/PIR runtime issues seen on Kaggle CPU runtime.
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_use_onednn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"
os.environ["DISABLE_MODEL_SOURCE_CHECK"] = "False"  # let PaddleOCR check/download models when Internet is ON

# Do not pip install PaddleOCR in notebook cells.
# Pin versions in Kaggle Add-ons -> Install Dependencies instead.
KAGGLE_DEPENDENCY_MANAGER_INSTALL_COMMANDS = """pip install --no-cache-dir paddlepaddle==3.2.0
pip install --no-cache-dir paddleocr==3.3.3
pip install --no-cache-dir langchain==0.3.27 langchain-community==0.3.27 langchain-text-splitters==0.3.11
pip install --no-cache-dir pymupdf pillow tqdm openpyxl gdown
"""

print("PaddleOCR runtime flags set.")
print("Use Kaggle Dependency Manager install commands for PaddleOCR versions.")


In [ ]:
# ============================================================
# 0A. Check OCR dependencies installed by Kaggle Dependency Manager
# ============================================================

import sys
import subprocess


def show_pkg(pkg):
    try:
        out = subprocess.check_output(
            [sys.executable, "-m", "pip", "show", pkg],
            text=True,
        )
        lines = []
        for line in out.splitlines():
            if line.startswith(("Name:", "Version:", "Location:")):
                lines.append(line)
        print("\n".join(lines))
    except Exception as e:
        print(f"{pkg}: not found or cannot inspect ({e})")

for pkg in [
    "paddlepaddle",
    "paddlepaddle-gpu",
    "paddleocr",
    "paddlex",
    "langchain",
    "langchain-community",
    "langchain-text-splitters",
    "pymupdf",
]:
    print("\n---", pkg)
    show_pkg(pkg)


In [ ]:
# ============================================================
# 0B. Optional PaddleOCR import sanity check
# ============================================================
# Default is False so Run All does not build the OCR model twice.
# Set RUN_PADDLEOCR_SANITY_CHECK=True only when debugging dependency issues.

RUN_PADDLEOCR_SANITY_CHECK = False

if RUN_PADDLEOCR_SANITY_CHECK:
    import os
    import importlib.util as _importlib_util

    os.environ["FLAGS_use_mkldnn"] = "0"
    os.environ["FLAGS_use_onednn"] = "0"
    os.environ["FLAGS_enable_pir_api"] = "0"
    os.environ.setdefault("DISABLE_MODEL_SOURCE_CHECK", "False")

    # Kaggle can have a Torch/NCCL mismatch. PaddleOCR does not need Torch for OCR.
    # Hide torch during PaddleOCR import to avoid: libtorch_cuda.so undefined symbol ncclCommShrink.
    _orig_find_spec = _importlib_util.find_spec
    def _find_spec_no_torch(name, *args, **kwargs):
        if name == "torch" or name.startswith("torch."):
            return None
        return _orig_find_spec(name, *args, **kwargs)

    import paddle
    print("paddle version:", paddle.__version__)
    print("paddle compiled with cuda:", paddle.is_compiled_with_cuda())
    try:
        print("paddle device:", paddle.device.get_device())
    except Exception as e:
        print("cannot get paddle device:", e)

    _importlib_util.find_spec = _find_spec_no_torch
    try:
        from paddleocr import PaddleOCR
        ocr_test = PaddleOCR(
            lang="vi",
            ocr_version="PP-OCRv5",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )
        print("PaddleOCR init OK.")
    finally:
        _importlib_util.find_spec = _orig_find_spec
else:
    print("Skip PaddleOCR sanity check. Cell 13A will build the OCR model once.")


In [ ]:
# ============================================================
# 0. Kaggle / local config
# ============================================================
from pathlib import Path
import os, sys, json, re, shutil, subprocess, unicodedata, zipfile, math, time
from collections import Counter, defaultdict

IS_KAGGLE = Path('/kaggle').exists()
WORKING_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1QZzyaozPRLcm5Y2nmUUbigyVFnsX_tkW'
DRIVE_INPUT_DIR = WORKING_DIR / 'drive_input'
INPUT_ROOTS = [DRIVE_INPUT_DIR, Path('/kaggle/input'), WORKING_DIR, Path.cwd(), Path('/mnt/data')]
INPUT_ROOTS = [p for p in INPUT_ROOTS if p.exists()]

# Optional: if your repo is already cloned, add it to sys.path.
# This notebook does not require repo code, but this keeps compatibility with your Kaggle setup.
REPO_CANDIDATES = [
    WORKING_DIR / 'SinoNom-NLP',
    WORKING_DIR / 'sinonom-nlp',
    WORKING_DIR / 'ocr-vietnam',
]
for repo in REPO_CANDIDATES:
    if repo.exists():
        sys.path.insert(0, str(repo))
        print('Added repo to sys.path:', repo)
        break

OUTPUT_DIR = WORKING_DIR / 'output_task2_dntc_full_pipeline'
DATA_DIR = OUTPUT_DIR / 'data'
REVIEW_DIR = OUTPUT_DIR / 'review'
CROP_DIR = REVIEW_DIR / 'crops'
FINAL_DIR = OUTPUT_DIR / 'final'
PACKAGE_DIR = OUTPUT_DIR / 'packages'

for d in [DATA_DIR, REVIEW_DIR, CROP_DIR, FINAL_DIR, PACKAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Main switches
FORCE_REEXTRACT = True
WRITE_XLSX = True                 # set False if openpyxl is slow
RENDER_CROPS = True               # render image crops for suspicious rows
MAX_CROPS = 5000                  # safety cap
REVIEW_SCORE_THRESHOLD = 3

# Optional secondary OCR for suspicious crops.
# Keep False for fast baseline. Set True when you want automatic OCR on suspicious crops.
ENABLE_SECONDARY_OCR = False
SECONDARY_OCR_ENGINE = 'easyocr'  # legacy; v5 uses PaddleOCR by default
SECONDARY_OCR_MAX_ROWS = 3000
SECONDARY_OCR_MIN_SCORE = 4

# Page range is 1-based and inclusive. None means all pages.
# Keys are dynamic work_id values generated from each PDF filename.
# Example: PAGE_RANGES = {'dai_nam_nhat_thong_chi_tap_01': (13, None)}
PAGE_RANGES = {}

print('IS_KAGGLE:', IS_KAGGLE)
print('WORKING_DIR:', WORKING_DIR)
print('DRIVE_FOLDER_URL:', DRIVE_FOLDER_URL)
print('DRIVE_INPUT_DIR:', DRIVE_INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('INPUT_ROOTS:', INPUT_ROOTS)

In [ ]:
# ============================================================
# 1. Install / import dependencies
# This cell does not install PaddleOCR. PaddleOCR must be pinned by Kaggle Dependency Manager.
# ============================================================
def pip_install(packages):
    if isinstance(packages, str):
        packages = [packages]
    print('Installing:', packages)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=False)

try:
    import fitz  # PyMuPDF
except Exception:
    pip_install(['pymupdf'])
    import fitz

try:
    import pandas as pd
except Exception:
    pip_install(['pandas'])
    import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    pip_install(['tqdm'])
    from tqdm.auto import tqdm

if WRITE_XLSX:
    try:
        import openpyxl
    except Exception:
        pip_install(['openpyxl'])
        import openpyxl

try:
    import gdown
except Exception:
    pip_install(['gdown'])
    import gdown

print('fitz/PyMuPDF:', fitz.__doc__.split('\n')[0])
print('pandas:', pd.__version__)

In [ ]:
# ============================================================
# 2. Download / locate PDF files from Google Drive
# ============================================================
def strip_accents(s: str) -> str:
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', s)

def norm_name(s: str) -> str:
    s = strip_accents(str(s)).lower()
    s = re.sub(r'[^a-z0-9]+', '', s)
    return s

def slugify_work_id(path: Path) -> str:
    stem = strip_accents(path.stem).lower()
    stem = re.sub(r'[^a-z0-9]+', '_', stem)
    stem = re.sub(r'_+', '_', stem).strip('_')
    return stem or f'pdf_{abs(hash(str(path))) % 1000000}'

def unique_work_id(base: str, used: set[str]) -> str:
    wid = base
    i = 2
    while wid in used:
        wid = f'{base}_{i}'
        i += 1
    used.add(wid)
    return wid

def download_drive_folder_if_needed():
    DRIVE_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    existing_pdfs = sorted(DRIVE_INPUT_DIR.rglob('*.pdf'))
    if existing_pdfs:
        print(f'Drive input already has {len(existing_pdfs)} PDF(s), skip download: {DRIVE_INPUT_DIR}')
        return

    print('Downloading Google Drive folder:')
    print(' ', DRIVE_FOLDER_URL)
    print('to:', DRIVE_INPUT_DIR)
    try:
        gdown.download_folder(
            url=DRIVE_FOLDER_URL,
            output=str(DRIVE_INPUT_DIR),
            quiet=False,
            use_cookies=False,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions do not support remaining_ok.
        gdown.download_folder(
            url=DRIVE_FOLDER_URL,
            output=str(DRIVE_INPUT_DIR),
            quiet=False,
            use_cookies=False,
        )

def find_pdfs(input_roots):
    candidates = []
    for root in input_roots:
        try:
            for p in root.rglob('*.pdf'):
                if p.is_file():
                    candidates.append(p)
        except Exception:
            pass
    # de-duplicate by resolved path string
    seen = set()
    out = []
    for p in candidates:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            out.append(p)
    return sorted(out, key=lambda p: str(p).lower())

download_drive_folder_if_needed()

pdf_candidates = find_pdfs([DRIVE_INPUT_DIR])
if not pdf_candidates:
    print('No PDF found in Drive input; fallback to all INPUT_ROOTS.')
    pdf_candidates = find_pdfs(INPUT_ROOTS)

print('PDF candidates:', len(pdf_candidates))
for p in pdf_candidates:
    print(' -', p.name, '->', p)

PDF_PATHS = {}
used_work_ids = set()
for p in pdf_candidates:
    wid = unique_work_id(slugify_work_id(p), used_work_ids)
    PDF_PATHS[wid] = p

print()
print('Mapped PDFs from filenames:')
for wid in sorted(PDF_PATHS):
    print(wid, '->', PDF_PATHS[wid].name)

PDF_MANIFEST = pd.DataFrame([
    {'work_id': wid, 'file_name': p.name, 'pdf_path': str(p), 'size_mb': p.stat().st_size / 1024 / 1024}
    for wid, p in sorted(PDF_PATHS.items())
])
manifest_path = DATA_DIR / 'pdf_manifest.csv'
PDF_MANIFEST.to_csv(manifest_path, index=False, encoding='utf-8-sig')
print()
print('Saved PDF manifest:', manifest_path)
PDF_MANIFEST

In [ ]:
# Optional manual override if auto mapping failed.
# Use work_id keys generated from filenames, for example:
# PDF_PATHS = {
#     'dai_nam_nhat_thong_chi_tap_01': Path('/kaggle/working/drive_input/book1.pdf'),
#     'another_pdf_name': Path('/kaggle/working/drive_input/folder/another.pdf'),
# }
# PAGE_RANGES = {
#     'dai_nam_nhat_thong_chi_tap_01': (13, None),
# }

assert PDF_PATHS, 'No PDF files found. Check DRIVE_FOLDER_URL, Drive sharing permissions, or Kaggle Internet.'
for wid, p in PDF_PATHS.items():
    assert p.exists(), f'Missing file for {wid}: {p}'

print(f'Ready to process {len(PDF_PATHS)} PDF(s).')
print('PAGE_RANGES keys available:', sorted(PDF_PATHS.keys()))

In [ ]:
# ============================================================
# 3. Utility functions: clean, classify, correct, split
# ============================================================
VIET_CHARS = set('ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ')
WEIRD_RE = re.compile(r'[^\w\sÀ-ỹ.,;:!?(){}\[\]"\'“”‘’/\-–—%]', re.UNICODE)

KNOWN_HEADINGS = {
    'DAI NAM NHAT THONG CHI': 'ĐẠI NAM NHẤT THỐNG CHÍ',
    'QUOC SU QUAN TRIEU NGUYEN': 'QUỐC SỬ QUÁN TRIỀU NGUYỄN',
    'VIEN KHOA HOC XA HOI VIET NAM': 'VIỆN KHOA HỌC XÃ HỘI VIỆT NAM',
    'VIEN SU HOC': 'VIỆN SỬ HỌC',
    'LOI NOI DAU': 'Lời nói đầu',
    'PHAN DA': 'PHẦN DÃ',
    'DUNG DAT VA DIEN CACH': 'DỰNG ĐẶT VÀ DIÊN CÁCH',
    'DUNG PAT VA DIEN CACH': 'DỰNG ĐẶT VÀ DIÊN CÁCH',
    'HINH THE': 'HÌNH THẾ',
    'KHI HAU': 'KHÍ HẬU',
    'PHONG TUC': 'PHONG TỤC',
    'THANH TRI': 'THÀNH TRÌ',
    'TRUONG HOC': 'TRƯỜNG HỌC',
    'HO KHAU': 'HỘ KHẨU',
    'THUE RUONG': 'THUẾ RUỘNG',
    'NUI SONG': 'NÚI SÔNG',
    'QUAN TAN': 'QUAN TẤN',
    'DICH TRAM': 'DỊCH TRẠM',
    'THI LAP': 'THỊ LẬP',
    'TU MIEU': 'TỪ MIẾU',
    'THO SAN': 'THỔ SẢN',
}

# Conservative global OCR replacements. Avoid aggressive semantic rewriting.
GLOBAL_REGEX_RULES = [
    (r'\bDAI\s*NAM\s*NHAT\s*THONG\s*CH[IÍ]\b', 'ĐẠI NAM NHẤT THỐNG CHÍ'),
    (r'\bNHAT\s*THONG\s*CH[IÍ]\b', 'NHẤT THỐNG CHÍ'),
    (r'\bQUOC\s*SU\s*QUAN\s*TRIEU\s*NGUYEN\b', 'QUỐC SỬ QUÁN TRIỀU NGUYỄN'),
    (r'\bVIEN\s*KHOA\s*HOC\s*XA\s*HOI\s*VIET\s*NAM\b', 'VIỆN KHOA HỌC XÃ HỘI VIỆT NAM'),
    (r'\bVIEN\s*SU\s*HOC\b', 'VIỆN SỬ HỌC'),
    (r'\bPHAN\s*DA\b', 'PHẦN DÃ'),
    (r'\bDUNG\s*(?:DAT|PAT)\s*VA\s*DIEN\s*CACH\b', 'DỰNG ĐẶT VÀ DIÊN CÁCH'),
    (r'\bHINH\s*THE\b', 'HÌNH THẾ'),
    (r'\bKHI\s*HAU\b', 'KHÍ HẬU'),
    (r'\bPHONG\s*TUC\b', 'PHONG TỤC'),
    (r'\bTHANH\s*TRI\b', 'THÀNH TRÌ'),
    (r'\bTRUONG\s*HOC\b', 'TRƯỜNG HỌC'),
    (r'\bHO\s*KHAU\b', 'HỘ KHẨU'),
    (r'\bTHUE\s*RUONG\b', 'THUẾ RUỘNG'),
    (r'\bNUI\s*SONG\b', 'NÚI SÔNG'),
    (r'\bQUYEN\b', 'QUYỂN'),
    (r'\bTINH\b', 'TỈNH'),
    (r'\bHUYEN\b', 'HUYỆN'),
    (r'\bPHU\b', 'PHỦ'),
    (r'\bdia\s+giới\b', 'địa giới'),
    (r'\btinh\s+([A-ZÀ-Ỹ])', r'tỉnh \1'),
    (r'\bdim\b', 'dặm'),
    (r'\bbi€n\b', 'biển'),
    (r'\bcudp\b', 'cướp'),
    (r'\blay\b', 'lấy'),
    (r'\blai\b', 'lại'),
    (r'\bdem\b', 'đem'),
    (r'\bva\b', 'và'),
]

ROMAN_RE = re.compile(r'^(QUY[ỂE]N|QUYEN)\s+[IVXLCDM]+$', re.IGNORECASE)

def normalize_unicode(s):
    s = '' if s is None or (isinstance(s, float) and math.isnan(s)) else str(s)
    s = unicodedata.normalize('NFC', s)
    # Common OCR noise chars
    s = s.replace('\uFFFE', ' ')
    s = s.replace('\u200b', '')
    s = s.replace('\ufeff', '')
    s = s.replace('￾', ' ')
    s = s.replace('ˆ', ' ')
    s = s.replace('©', ' ')
    s = s.replace('®', ' ')
    s = s.replace('™', ' ')
    s = s.replace('¦', ' ')
    s = s.replace('|', ' ')
    s = s.replace('`', ' ')
    s = s.replace('´', "'")
    return s

def normalize_space(s):
    s = normalize_unicode(s)
    s = s.replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def remove_accents_upper(s):
    s = strip_accents(normalize_space(s)).upper()
    s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def is_page_number_line(s):
    s = normalize_space(s)
    return bool(re.fullmatch(r'[-–—]?[ ]*\d{1,4}[ ]*[-–—]?', s))

def is_file_noise_line(s):
    s0 = normalize_space(s).lower()
    return s0.endswith('.pdf') or '.pdf' in s0 or s0 in {'le.pdf', 'ch.pdf'}

def is_probable_heading(s):
    s = normalize_space(s)
    if not s:
        return False
    key = remove_accents_upper(s)
    if key in KNOWN_HEADINGS:
        return True
    if ROMAN_RE.match(key):
        return True
    if re.fullmatch(r'(TINH|TỈNH|HUYEN|HUYỆN|PHU|PHỦ|CHAU|CHÂU)\s+[A-ZÀ-Ỹ\s\-]+', s.upper()):
        return True
    # Short all-caps line with very few lowercase chars
    letters = re.findall(r'[A-Za-zÀ-ỹ]', s)
    if 2 <= len(s) <= 45 and letters:
        lower = sum(ch.islower() for ch in letters)
        upper = sum(ch.isupper() for ch in letters)
        if upper >= max(3, lower * 3) and not re.search(r'[.,;]$', s):
            return True
    return False

def fix_heading(s):
    raw = normalize_space(s)
    key = remove_accents_upper(raw)
    if key in KNOWN_HEADINGS:
        return KNOWN_HEADINGS[key]
    # Fix common heading prefixes
    x = raw
    for pat, repl in GLOBAL_REGEX_RULES[:16]:
        x = re.sub(pat, repl, x, flags=re.IGNORECASE)
    x = re.sub(r'\bQUYEN\b', 'QUYỂN', x, flags=re.IGNORECASE)
    x = re.sub(r'\bTINH\b', 'TỈNH', x, flags=re.IGNORECASE)
    return normalize_space(x)

def conservative_correct_text(s, text_type='paragraph'):
    raw = normalize_space(s)
    if not raw:
        return raw, []
    out = raw
    changes = []

    if text_type == 'heading' or is_probable_heading(out):
        fixed = fix_heading(out)
        if fixed != out:
            changes.append('fix_heading')
            out = fixed

    # cleanup punctuation and spacing
    before = out
    out = normalize_unicode(out)
    out = re.sub(r'([0-9])([A-Za-zÀ-ỹ])', r'\1 \2', out)
    out = re.sub(r'([A-Za-zÀ-ỹ])([0-9])', r'\1 \2', out)
    out = re.sub(r'\s+([,.;:!?])', r'\1', out)
    out = re.sub(r'([,.;:!?])([^\s,.;:!?])', r'\1 \2', out)
    out = re.sub(r'\(\s+', '(', out)
    out = re.sub(r'\s+\)', ')', out)
    out = re.sub(r'\s+', ' ', out).strip()
    if out != before:
        changes.append('normalize_spacing_punct')

    # Apply conservative regex rules
    for pat, repl in GLOBAL_REGEX_RULES:
        before = out
        out = re.sub(pat, repl, out, flags=re.IGNORECASE)
        if out != before:
            changes.append(f'rule:{pat}')

    # Fix common accidental separators but do not overcorrect names
    before = out
    out = out.replace(' . ', '. ')
    out = out.replace(' , ', ', ')
    out = re.sub(r'\s+', ' ', out).strip()
    if out != before:
        changes.append('final_spacing')
    return out, changes

def weird_char_count(s):
    return len(WEIRD_RE.findall(normalize_space(s)))

def vietnamese_accent_count(s):
    return sum(1 for ch in normalize_space(s) if ch in VIET_CHARS)

def ascii_word_ratio(s):
    words = re.findall(r'[A-Za-zÀ-ỹ]+', normalize_space(s))
    if not words:
        return 0.0
    ascii_words = [w for w in words if all(ord(ch) < 128 for ch in w)]
    return len(ascii_words) / len(words)

def digit_tokens(s):
    return re.findall(r'\d+(?:[.,]\d+)?', normalize_space(s))

def suspicious_score(s, text_type='sentence'):
    s = normalize_space(s)
    score = 0
    reasons = []
    if not s:
        return 99, ['empty']
    wc = weird_char_count(s)
    if wc:
        add = min(5, wc)
        score += add
        reasons.append(f'weird_chars={wc}')
    if re.search(r'[€ñ￾©®ˆ|]', s):
        score += 4
        reasons.append('known_noise_char')
    if re.search(r'\b[A-Za-z]{2,}\d|\d[A-Za-zÀ-ỹ]{2,}\b', s):
        score += 2
        reasons.append('digit_letter_stuck')
    if re.search(r'[A-Za-zÀ-ỹ]{28,}', s):
        score += 2
        reasons.append('very_long_token')
    ar = ascii_word_ratio(s)
    # high ASCII ratio is suspicious only for longer Vietnamese prose, not Roman headings
    if len(s) > 30 and ar > 0.55 and vietnamese_accent_count(s) < max(4, len(s) * 0.02):
        score += 2
        reasons.append(f'high_ascii_ratio={ar:.2f}')
    if len(s) > 280 and not re.search(r'[.!?…]$', s):
        score += 1
        reasons.append('long_no_sentence_end')
    if len(s) < 4:
        score += 1
        reasons.append('too_short')
    if text_type == 'heading' and ascii_word_ratio(s) > 0.5:
        score += 1
        reasons.append('heading_possible_missing_accents')
    return score, reasons

# Protect abbreviations before sentence split
PROTECT = {
    'tr.C.N.': 'tr<CN>',
    'v.v..': 'vv<DOTDOT>',
    'v.v.': 'vv<DOT>',
}
UNPROTECT = {v: k for k, v in PROTECT.items()}

def protect_abbrev(s):
    for k, v in PROTECT.items():
        s = s.replace(k, v)
    return s

def unprotect_abbrev(s):
    for k, v in UNPROTECT.items():
        s = s.replace(k, v)
    return s

def split_long_sentence(s, max_len=430):
    s = normalize_space(s)
    if len(s) <= max_len:
        return [s]
    parts = []
    # Split on semicolons before common discourse markers in this historical text
    chunks = re.split(r';\s+(?=(?:năm|đời|phía|sau|trước|lại|nay|xưa|thời|đầu|cuối)\b)', s, flags=re.IGNORECASE)
    cur = ''
    for ch in chunks:
        ch = ch.strip()
        if not ch:
            continue
        candidate = (cur + '; ' + ch).strip('; ') if cur else ch
        if len(candidate) <= max_len or not cur:
            cur = candidate
        else:
            parts.append(cur)
            cur = ch
    if cur:
        parts.append(cur)
    return parts

def split_sentences(text, text_type='paragraph'):
    text = normalize_space(text)
    if not text:
        return []
    if text_type in {'heading', 'footnote'}:
        return [text]
    t = protect_abbrev(text)
    # standard sentence ends
    pieces = re.split(r'(?<=[.!?…])\s+(?=["“‘\(\[]?[A-ZÀ-ỸĐ0-9])', t)
    out = []
    for p in pieces:
        p = unprotect_abbrev(normalize_space(p))
        if not p:
            continue
        out.extend(split_long_sentence(p))
    # merge tiny fragments
    merged = []
    for p in out:
        if merged and len(p) < 18 and not re.search(r'[.!?…]$', merged[-1]):
            merged[-1] = normalize_space(merged[-1] + ' ' + p)
        else:
            merged.append(p)
    return merged

print('Utilities ready')

In [ ]:
# ============================================================
# 4. Extract lines and group into paragraphs
# ============================================================
def union_bbox(boxes):
    xs0, ys0, xs1, ys1 = zip(*boxes)
    return [min(xs0), min(ys0), max(xs1), max(ys1)]

def line_from_spans(line):
    spans = line.get('spans', [])
    texts = []
    bboxes = []
    sizes = []
    for sp in spans:
        txt = sp.get('text', '')
        if txt:
            texts.append(txt)
            bboxes.append(sp.get('bbox', line.get('bbox', [0,0,0,0])))
            sizes.append(sp.get('size', 0))
    text = normalize_space(''.join(texts))
    if not text:
        return None
    bbox = union_bbox(bboxes) if bboxes else list(line.get('bbox', [0,0,0,0]))
    return {'text': text, 'bbox': bbox, 'font_size': sum(sizes)/len(sizes) if sizes else 0}

def extract_page_lines(page):
    data = page.get_text('dict')
    lines = []
    for block in data.get('blocks', []):
        if block.get('type', 0) != 0:
            continue
        for line in block.get('lines', []):
            item = line_from_spans(line)
            if not item:
                continue
            txt = item['text']
            if is_page_number_line(txt) or is_file_noise_line(txt):
                continue
            if len(txt) == 1 and not re.match(r'[A-Za-zÀ-ỹ0-9]', txt):
                continue
            lines.append(item)
    lines.sort(key=lambda x: (round(x['bbox'][1], 1), round(x['bbox'][0], 1)))
    return lines

def paragraph_type_for_line(text, page_height=None, bbox=None):
    s = normalize_space(text)
    if is_probable_heading(s):
        return 'heading'
    if re.match(r'^\(?\d{1,3}\)\s+', s) or re.match(r'^\d{1,3}\)\s+', s):
        return 'footnote'
    if page_height and bbox and bbox[1] > page_height * 0.82 and re.match(r'^[\(\[]?\d{1,3}[\)\]]', s):
        return 'footnote'
    return 'paragraph'

def group_lines_to_paragraphs(lines, page_rect):
    paragraphs = []
    current = []
    current_boxes = []
    current_type = None
    prev_y = None
    prev_h = None
    page_height = page_rect.height

    def flush():
        nonlocal current, current_boxes, current_type
        if current:
            raw = normalize_space(' '.join(current))
            if raw:
                bbox = union_bbox(current_boxes)
                paragraphs.append({'type': current_type or 'paragraph', 'raw_text': raw, 'bbox': bbox})
        current = []
        current_boxes = []
        current_type = None

    for ln in lines:
        text = ln['text']
        bbox = ln['bbox']
        typ = paragraph_type_for_line(text, page_height, bbox)
        y0, y1 = bbox[1], bbox[3]
        h = max(1.0, y1 - y0)
        gap = 0 if prev_y is None else y0 - prev_y
        large_gap = prev_h is not None and gap > max(8, prev_h * 0.95)

        if typ == 'heading':
            flush()
            paragraphs.append({'type': 'heading', 'raw_text': normalize_space(text), 'bbox': bbox})
        elif typ == 'footnote':
            flush()
            current = [text]
            current_boxes = [bbox]
            current_type = 'footnote'
        else:
            if current_type in {'heading', 'footnote'}:
                flush()
            # start new paragraph when vertical gap is large or line appears indented after sentence end
            should_new = False
            if current and large_gap:
                should_new = True
            if current:
                prev_text = current[-1]
                if re.search(r'[.!?…:]$', prev_text) and bbox[0] > (current_boxes[-1][0] + 18):
                    should_new = True
            if should_new:
                flush()
            if not current:
                current_type = 'paragraph'
            current.append(text)
            current_boxes.append(bbox)
        prev_y = y1
        prev_h = h
    flush()
    return paragraphs

def page_in_range(page_number_1based, range_tuple):
    if range_tuple is None:
        return True
    start, end = range_tuple
    if start is not None and page_number_1based < start:
        return False
    if end is not None and page_number_1based > end:
        return False
    return True

def extract_all_paragraphs(pdf_paths, page_ranges):
    page_records = []
    para_records = []
    for work_id in sorted(pdf_paths):
        pdf_path = Path(pdf_paths[work_id])
        doc = fitz.open(str(pdf_path))
        prange = page_ranges.get(work_id)
        print(f'{work_id}: {pdf_path.name}, pages={doc.page_count}, range={prange}')
        for page_idx in tqdm(range(doc.page_count), desc=f'extract {work_id}'):
            page_number = page_idx + 1
            if not page_in_range(page_number, prange):
                continue
            page = doc[page_idx]
            rect = page.rect
            lines = extract_page_lines(page)
            paragraphs = group_lines_to_paragraphs(lines, rect)
            raw_page_text = '\n'.join([p['raw_text'] for p in paragraphs])
            page_records.append({
                'work_id': work_id,
                'pdf_path': str(pdf_path),
                'page_idx': page_idx,
                'page_number': page_number,
                'width': rect.width,
                'height': rect.height,
                'line_count': len(lines),
                'paragraph_count': len(paragraphs),
                'raw_text': raw_page_text,
            })
            for local_i, p in enumerate(paragraphs):
                auto, changes = conservative_correct_text(p['raw_text'], p['type'])
                score, reasons = suspicious_score(auto, p['type'])
                para_records.append({
                    'paragraph_id': f'{work_id}_p{page_number:04d}_{local_i:03d}',
                    'work_id': work_id,
                    'pdf_path': str(pdf_path),
                    'page_idx': page_idx,
                    'page_number': page_number,
                    'paragraph_index_on_page': local_i,
                    'type': p['type'],
                    'bbox': json.dumps([round(float(x), 2) for x in p['bbox']]),
                    'raw_text': p['raw_text'],
                    'auto_text': auto,
                    'correction_rules': '|'.join(changes),
                    'quality_score': score,
                    'reasons': '|'.join(reasons),
                })
        doc.close()
    return page_records, para_records

pages_path = DATA_DIR / 'pages.jsonl'
paras_path = DATA_DIR / 'paragraphs.jsonl'

if FORCE_REEXTRACT or not paras_path.exists():
    page_records, para_records = extract_all_paragraphs(PDF_PATHS, PAGE_RANGES)
    with open(pages_path, 'w', encoding='utf-8') as f:
        for r in page_records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    with open(paras_path, 'w', encoding='utf-8') as f:
        for r in para_records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
else:
    page_records = [json.loads(x) for x in open(pages_path, encoding='utf-8')]
    para_records = [json.loads(x) for x in open(paras_path, encoding='utf-8')]

print('Pages:', len(page_records))
print('Paragraphs:', len(para_records))
print('Saved:', pages_path)
print('Saved:', paras_path)

In [ ]:
# ============================================================
# 5. Sentence segmentation
# ============================================================
def build_sentences(para_records):
    sent_records = []
    seq_by_work = defaultdict(int)
    for p in tqdm(para_records, desc='split sentences'):
        text = p.get('auto_text', '')
        parts = split_sentences(text, p.get('type', 'paragraph'))
        if not parts:
            continue
        for i, sent in enumerate(parts):
            sent = normalize_space(sent)
            score, reasons = suspicious_score(sent, 'sentence' if p.get('type') == 'paragraph' else p.get('type'))
            wid = p['work_id']
            seq_by_work[wid] += 1
            sent_records.append({
                'sent_id': f'{wid}_s{seq_by_work[wid]:06d}',
                'paragraph_id': p['paragraph_id'],
                'work_id': wid,
                'pdf_path': p['pdf_path'],
                'page_idx': p['page_idx'],
                'page_number': p['page_number'],
                'sentence_index_in_paragraph': i,
                'type': p['type'],
                'bbox': p['bbox'],
                'raw_paragraph_text': p['raw_text'],
                'auto_paragraph_text': p['auto_text'],
                'raw_text': sent,          # sentence after paragraph-level correction
                'auto_text': sent,
                'quality_score': score,
                'reasons': '|'.join(reasons),
                'correction_rules': p.get('correction_rules', ''),
            })
    return sent_records

sent_records = build_sentences(para_records)
sent_df = pd.DataFrame(sent_records)
sent_csv = DATA_DIR / 'sentences.csv'
sent_jsonl = DATA_DIR / 'sentences.jsonl'
sent_df.to_csv(sent_csv, index=False, encoding='utf-8-sig')
with open(sent_jsonl, 'w', encoding='utf-8') as f:
    for r in sent_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('Sentences:', len(sent_df))
print('Saved:', sent_csv)
print('Saved:', sent_jsonl)
sent_df.head(10)

In [ ]:
# ============================================================
# 6. Render crop images for suspicious sentences
# ============================================================
def parse_bbox(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return None
    if isinstance(x, (list, tuple)):
        return list(x)
    return None

def render_pdf_crop(pdf_path, page_idx, bbox, out_path, zoom=2.2, pad=12):
    pdf_path = Path(pdf_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        doc = fitz.open(str(pdf_path))
        page = doc[int(page_idx)]
        rect = page.rect
        if bbox is None:
            clip = rect
        else:
            x0, y0, x1, y1 = [float(v) for v in bbox]
            clip = fitz.Rect(
                max(rect.x0, x0 - pad),
                max(rect.y0, y0 - pad),
                min(rect.x1, x1 + pad),
                min(rect.y1, y1 + pad),
            )
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, clip=clip, alpha=False)
        pix.save(str(out_path))
        doc.close()
        return str(out_path)
    except Exception as e:
        return ''

review_df = sent_df.copy()
review_df['manual_text'] = ''
review_df['review_status'] = ''
review_df['crop_path'] = ''
review_df['suggestion'] = review_df['auto_text']

sus_mask = review_df['quality_score'].fillna(0).astype(float) >= REVIEW_SCORE_THRESHOLD
# Always include headings with issues, and all rows that changed by rules if suspicious.
review_rows = review_df[sus_mask].copy()
print('Suspicious rows:', len(review_rows), 'threshold=', REVIEW_SCORE_THRESHOLD)

if RENDER_CROPS:
    crop_count = 0
    crop_paths = {}
    for idx, row in tqdm(review_rows.iterrows(), total=min(len(review_rows), MAX_CROPS), desc='render crops'):
        if crop_count >= MAX_CROPS:
            break
        bbox = parse_bbox(row.get('bbox'))
        out_name = f"{row['work_id']}_p{int(row['page_number']):04d}_{row['sent_id']}.png"
        out_path = CROP_DIR / row['work_id'] / out_name
        crop = render_pdf_crop(row['pdf_path'], int(row['page_idx']), bbox, out_path)
        crop_paths[idx] = crop
        crop_count += 1
    for idx, cp in crop_paths.items():
        review_df.at[idx, 'crop_path'] = cp

suspicious_review = review_df[sus_mask].copy()
review_csv = REVIEW_DIR / 'suspicious_review.csv'
review_xlsx = REVIEW_DIR / 'suspicious_review.xlsx'
suspicious_review.to_csv(review_csv, index=False, encoding='utf-8-sig')
print('Saved:', review_csv)

if WRITE_XLSX:
    try:
        suspicious_review.to_excel(review_xlsx, index=False)
        print('Saved:', review_xlsx)
    except Exception as e:
        print('Could not write xlsx:', e)

suspicious_review[['sent_id','work_id','page_number','quality_score','reasons','auto_text','crop_path']].head(20)

In [ ]:
# ============================================================
# 7. Spell / weird token candidates
# ============================================================
def token_candidates(texts):
    counter = Counter()
    examples = {}
    for s in texts:
        s = normalize_space(s)
        toks = re.findall(r'[A-Za-zÀ-ỹ0-9€ñ￾©®ˆ|]{2,}', s)
        for t in toks:
            suspicious = False
            if weird_char_count(t) > 0:
                suspicious = True
            if re.search(r'[A-Za-zÀ-ỹ]\d|\d[A-Za-zÀ-ỹ]', t):
                suspicious = True
            if len(t) > 24:
                suspicious = True
            if len(t) >= 4 and all(ord(ch) < 128 for ch in t) and t.lower() not in {'nam','namky','bac','kinh','long','phu','son','hai'}:
                # not all ASCII words are wrong, but list them for review when common
                suspicious = True
            if suspicious:
                counter[t] += 1
                examples.setdefault(t, s[:240])
    rows = []
    for tok, cnt in counter.most_common(1000):
        rows.append({'token': tok, 'count': cnt, 'example': examples.get(tok, '')})
    return pd.DataFrame(rows)

cand_df = token_candidates(sent_df['auto_text'].tolist())
spell_csv = REVIEW_DIR / 'spell_candidates.csv'
cand_df.to_csv(spell_csv, index=False, encoding='utf-8-sig')
print('Saved:', spell_csv)
cand_df.head(30)

In [ ]:
# ============================================================
# 8. Optional secondary OCR for suspicious crops
# ============================================================
def similarity(a, b):
    from difflib import SequenceMatcher
    a = normalize_space(a)
    b = normalize_space(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def choose_better_text(base_text, ocr_text, ocr_score):
    base = normalize_space(base_text)
    ocr = normalize_space(ocr_text)
    if not ocr:
        return base, 'no_ocr'
    if len(ocr) < 0.45 * max(1, len(base)):
        return base, 'reject_ocr_too_short'

    base_digits = digit_tokens(base)
    ocr_digits = digit_tokens(ocr)
    if base_digits:
        missing = [d for d in base_digits if d not in ocr_digits]
        if len(missing) >= max(1, len(base_digits) // 2):
            return base, 'reject_ocr_lost_digits'

    base_weird = weird_char_count(base)
    ocr_weird = weird_char_count(ocr)
    base_acc = vietnamese_accent_count(base)
    ocr_acc = vietnamese_accent_count(ocr)
    sim = similarity(base, ocr)

    if (
        ocr_score >= 0.35 and sim >= 0.35 and
        ocr_weird <= base_weird and ocr_acc >= base_acc and
        len(ocr) >= 0.65 * max(1, len(base))
    ):
        return ocr, f'accept_ocr(score={ocr_score:.2f},sim={sim:.2f},weird={base_weird}->{ocr_weird},accent={base_acc}->{ocr_acc})'

    if base_weird >= 2 and ocr_weird < base_weird and len(ocr) >= 0.60 * max(1, len(base)) and ocr_score >= 0.25:
        return ocr, f'accept_ocr_cleaner(score={ocr_score:.2f},sim={sim:.2f})'

    return base, f'keep_base(score={ocr_score:.2f},sim={sim:.2f},weird={base_weird}->{ocr_weird},accent={base_acc}->{ocr_acc})'

ocr_review_csv = REVIEW_DIR / 'suspicious_review_with_ocr.csv'

if ENABLE_SECONDARY_OCR:
    if SECONDARY_OCR_ENGINE.lower() != 'easyocr':
        raise ValueError('Only easyocr is implemented in this notebook.')
    try:
        import easyocr
    except Exception:
        pip_install(['easyocr'])
        import easyocr

    # Use GPU when Kaggle GPU is enabled; EasyOCR falls back if not.
    try:
        reader = easyocr.Reader(['vi', 'en'], gpu=True)
    except Exception:
        reader = easyocr.Reader(['vi', 'en'], gpu=False)

    def ocr_crop(crop_path):
        crop_path = normalize_space(crop_path)
        if not crop_path or not Path(crop_path).exists():
            return '', 0.0
        try:
            result = reader.readtext(crop_path, detail=1, paragraph=True)
        except Exception:
            return '', 0.0
        texts, scores = [], []
        for item in result:
            if len(item) >= 2:
                texts.append(normalize_space(item[1]))
            if len(item) >= 3:
                try:
                    scores.append(float(item[2]))
                except Exception:
                    pass
        return normalize_space(' '.join(texts)), (sum(scores) / len(scores) if scores else 0.0)

    ocr_df = pd.read_csv(review_csv)
    ocr_df['secondary_ocr_text'] = ''
    ocr_df['secondary_ocr_score'] = 0.0
    ocr_df['auto_choose_text'] = ocr_df['auto_text'].fillna('').astype(str)
    ocr_df['auto_choose_reason'] = 'not_selected'

    rows = ocr_df[
        (ocr_df['quality_score'].fillna(0).astype(float) >= SECONDARY_OCR_MIN_SCORE) &
        (ocr_df['crop_path'].fillna('').astype(str).str.len() > 0)
    ].index.tolist()[:SECONDARY_OCR_MAX_ROWS]

    print('Secondary OCR rows:', len(rows))
    for k, idx in enumerate(tqdm(rows, desc='secondary OCR'), 1):
        base_text = normalize_space(ocr_df.at[idx, 'auto_text'])
        crop_path = normalize_space(ocr_df.at[idx, 'crop_path'])
        ocr_text, ocr_score = ocr_crop(crop_path)
        chosen, reason = choose_better_text(base_text, ocr_text, ocr_score)
        ocr_df.at[idx, 'secondary_ocr_text'] = ocr_text
        ocr_df.at[idx, 'secondary_ocr_score'] = ocr_score
        ocr_df.at[idx, 'auto_choose_text'] = chosen
        ocr_df.at[idx, 'auto_choose_reason'] = reason

    ocr_df.to_csv(ocr_review_csv, index=False, encoding='utf-8-sig')
    print('Saved:', ocr_review_csv)
else:
    print('Secondary OCR disabled. Set ENABLE_SECONDARY_OCR=True in config to run it.')

In [ ]:
# ============================================================
# 9. Apply correction and export final files
# ============================================================
def read_review_file():
    candidates = [
        REVIEW_DIR / 'suspicious_review_with_ocr_corrected.csv',
        REVIEW_DIR / 'suspicious_review_corrected.csv',
        REVIEW_DIR / 'suspicious_review_corrected.xlsx',
        REVIEW_DIR / 'suspicious_review_with_ocr.csv',
        REVIEW_DIR / 'suspicious_review.csv',
    ]
    for p in candidates:
        if p.exists():
            print('Using review file:', p)
            if p.suffix.lower() == '.xlsx':
                return pd.read_excel(p), p
            return pd.read_csv(p), p
    print('No review file found; final uses baseline sentences.')
    return None, None

def nonempty(x):
    if x is None:
        return False
    if isinstance(x, float) and math.isnan(x):
        return False
    return bool(str(x).strip())

base = pd.read_csv(sent_csv)
review, review_path_used = read_review_file()

# Start from baseline auto text
base['final_text'] = base['auto_text'].fillna('').astype(str)
base['final_source'] = 'auto_text'
base['final_note'] = ''

if review is not None and 'sent_id' in review.columns:
    review = review.copy()
    review_by_id = review.set_index('sent_id', drop=False)
    for idx, row in base.iterrows():
        sid = row['sent_id']
        if sid not in review_by_id.index:
            continue
        rr = review_by_id.loc[sid]
        # handle duplicate index returning DataFrame
        if isinstance(rr, pd.DataFrame):
            rr = rr.iloc[0]
        chosen = None
        source = None
        if 'manual_text' in rr and nonempty(rr.get('manual_text')):
            chosen = str(rr.get('manual_text')).strip()
            source = 'manual_text'
        elif 'auto_choose_text' in rr and nonempty(rr.get('auto_choose_text')) and str(rr.get('auto_choose_text')).strip() != str(rr.get('auto_text', '')).strip():
            chosen = str(rr.get('auto_choose_text')).strip()
            source = 'secondary_ocr_auto_choose'
        elif 'auto_text' in rr and nonempty(rr.get('auto_text')):
            chosen = str(rr.get('auto_text')).strip()
            source = 'review_auto_text'
        if chosen is not None:
            base.at[idx, 'final_text'] = normalize_space(chosen)
            base.at[idx, 'final_source'] = source
            if 'auto_choose_reason' in rr:
                base.at[idx, 'final_note'] = str(rr.get('auto_choose_reason', ''))

# Final safety cleanup
base['final_text'] = base['final_text'].map(normalize_space)

final_csv = FINAL_DIR / 'final_sentences.csv'
final_jsonl = FINAL_DIR / 'final_sentences.jsonl'
base.to_csv(final_csv, index=False, encoding='utf-8-sig')
with open(final_jsonl, 'w', encoding='utf-8') as f:
    for r in base.to_dict(orient='records'):
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Export one txt per work_id
texts_dir = FINAL_DIR / 'texts'
texts_dir.mkdir(exist_ok=True, parents=True)
for wid, g in base.groupby('work_id'):
    out_txt = texts_dir / f'{wid}_final.txt'
    with open(out_txt, 'w', encoding='utf-8') as f:
        last_page = None
        for _, row in g.sort_values(['page_number', 'sent_id']).iterrows():
            if row['page_number'] != last_page:
                if last_page is not None:
                    f.write('\n')
                f.write(f'\n\n[Page {int(row["page_number"])}]\n')
                last_page = row['page_number']
            if row['type'] == 'heading':
                f.write('\n' + row['final_text'] + '\n')
            else:
                f.write(row['final_text'] + '\n')
    print('Saved:', out_txt)

print('Saved:', final_csv)
print('Saved:', final_jsonl)
print('Rows:', len(base))
base[['sent_id','work_id','page_number','type','final_source','final_text']].head(20)

In [ ]:
# ============================================================
# 10. Summary + zip package
# ============================================================
summary_rows = []
for wid, g in base.groupby('work_id'):
    summary_rows.append({
        'work_id': wid,
        'sentences': len(g),
        'pages': g['page_number'].nunique(),
        'suspicious_rows': int((g['quality_score'].fillna(0).astype(float) >= REVIEW_SCORE_THRESHOLD).sum()),
        'manual_or_ocr_changed': int((g['final_source'] != 'auto_text').sum()),
        'avg_quality_score': float(g['quality_score'].fillna(0).mean()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / 'summary.csv'
summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
print('Saved:', summary_csv)
display(summary_df)

# top remaining weird tokens in final output
remaining_df = token_candidates(base['final_text'].tolist())
remaining_csv = REVIEW_DIR / 'top_remaining_weird_tokens.csv'
remaining_df.to_csv(remaining_csv, index=False, encoding='utf-8-sig')
print('Saved:', remaining_csv)
display(remaining_df.head(30))

zip_path = PACKAGE_DIR / 'dntc_full_pipeline_output.zip'
if zip_path.exists():
    zip_path.unlink()

# Do not include the zip inside itself.
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob('*'):
        if p.is_file() and p != zip_path:
            z.write(p, p.relative_to(OUTPUT_DIR))

print('ZIP:', zip_path)
print('ZIP size MB:', round(zip_path.stat().st_size / 1024 / 1024, 2))

In [ ]:
# ============================================================
# 13A. Strong OCR v8 - PaddleOCR stable smoke test (Run All friendly)
# ============================================================
# This section keeps the original PDF text-layer pipeline.
# PaddleOCR is only used as a secondary OCR source for suspicious rows.
# It does NOT blindly replace base text.
#
# Requirements should be installed by Kaggle Dependency Manager, not by this cell.
# This cell is safe for Run All: it builds PaddleOCR once and runs a smoke test before Cell 13B.

import os
import re
import sys
import ast
import json
import math
import shutil
import zipfile
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_use_onednn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"
os.environ["DISABLE_MODEL_SOURCE_CHECK"] = "False"

import pandas as pd
import fitz
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
from tqdm.auto import tqdm

# ---------------- Config ----------------
ENABLE_STRONG_OCR_V8 = True
V8_STRONG_OCR_ENGINE = "paddleocr"

# Do not force GPU by default. GPU speeds up OCR but does not fix Vietnamese accents by itself.
# If your dependency package is paddlepaddle-gpu and Kaggle GPU is enabled, you can set this True.
V8_FORCE_GPU_IF_AVAILABLE = False

# Render settings. Larger padding helps preserve Vietnamese accents around the line.
V8_ZOOM = 5.0
V8_PAD_PT = 20.0
V8_SMOKE_TEST_ROWS = 12
V8_MAX_ROWS = 8000
V8_MIN_QUALITY_SCORE = 3
# Sentence rows currently inherit paragraph bboxes, so PaddleOCR crops can be wider than the target sentence.
# Keep auto-accept off by default; Cell 13B still writes OCR audit files for manual inspection.
V8_AUTO_ACCEPT = False
V8_ACCEPT_MIN_SIM = 0.82
V8_ACCEPT_BAD_MIN_SIM = 0.72
V8_MAX_OCR_LENGTH_RATIO = 1.35
V8_DEBUG_RAW_LIMIT = 5

# Run multiple crop/preprocess variants; choose the best OCR result conservatively.
V8_USE_CROP_VARIANTS = True

# ---------------- Paths ----------------
BASE_OUTPUT_DIR = Path(OUTPUT_DIR) if "OUTPUT_DIR" in globals() else Path("/kaggle/working/output_task2_dntc_full_pipeline")

V8_DIR = BASE_OUTPUT_DIR / "final_strong_ocr_v8_paddle"
V8_FINAL_DIR = V8_DIR / "final"
V8_TEXT_DIR = V8_FINAL_DIR / "texts"
V8_REVIEW_DIR = V8_DIR / "review"
V8_CROP_DIR = V8_DIR / "strong_ocr_crops"
V8_PACKAGE_DIR = V8_DIR / "packages"

for d in [V8_FINAL_DIR, V8_TEXT_DIR, V8_REVIEW_DIR, V8_CROP_DIR, V8_PACKAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR)
print("V8_DIR:", V8_DIR)

# ---------------- Load best available CSV ----------------
input_candidates = [
    BASE_OUTPUT_DIR / "final_strong_ocr_v7_gpu" / "final" / "final_sentences_strong_ocr_v7_gpu.csv",
    BASE_OUTPUT_DIR / "final_strong_ocr_v6" / "final" / "final_sentences_strong_ocr_v6.csv",
    BASE_OUTPUT_DIR / "final_postcorrected_v4" / "final" / "final_sentences_postcorrected_v4.csv",
    BASE_OUTPUT_DIR / "final_strong_ocr_v5" / "final" / "final_sentences_strong_ocr_v5.csv",
    BASE_OUTPUT_DIR / "final" / "final_sentences.csv",
    BASE_OUTPUT_DIR / "data" / "sentences.csv",
]

INPUT_SENT_CSV = None
for p in input_candidates:
    if p.exists():
        INPUT_SENT_CSV = p
        break

assert INPUT_SENT_CSV is not None, f"Cannot find input CSV. Tried: {input_candidates}"

df_v8 = pd.read_csv(INPUT_SENT_CSV)
print("Loaded:", INPUT_SENT_CSV)
print("Rows:", len(df_v8))
print("Columns:", list(df_v8.columns))

# ---------------- Text helpers ----------------
VIET_CHARS_V8 = set(
    "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩị"
    "óòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"
    "ĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊ"
    "ÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
)

WEIRD_RE_V8 = re.compile(r"[^\w\sÀ-ỹ.,;:!?(){}\[\]\"'“”‘’/\-–—%]", re.UNICODE)
NON_VIET_LATIN_RE_V8 = re.compile(r"[Α-ωА-яЁё]")

BAD_OCR_PATTERNS_V8 = [
    r"[€#ñ￾]",
    r"\bth[eéế]\s+k[yỷ]\b",
    r"\bdoi\s+#?ự\b",
    r"\bNHẤT\s+THONG\b",
    r"\bĐẠI\s+NAM\s+NHẬT\b",
    r"\bDAI\s+NAM\b",
    r"\bVIEN\s+KHOA\b",
    r"\bPHAM\s+TR[OỌ]NG\b",
    r"\bMinh\s+M[ée]nh\b",
    r"\bThiệu\s+Tri\b",
    r"\bHién\s+Tông\b",
    r"\bCao\s+Mén\b",
    r"\bNii\b",
    r"\bPEN\b",
    r"\bPAI\b",
    r"\bPHỦ\s+DE\b",
    r"\bbi€n\b|\bki€m\b|\bmi€u\b|\bchi€m\b",
    r"[A-Za-zÀ-ỹ]\d|\d[A-Za-zÀ-ỹ]",
    r"\b[A-Z]{2,}[a-zà-ỹ]+",
]

BAD_OCR_RE_V8 = re.compile("|".join(f"(?:{p})" for p in BAD_OCR_PATTERNS_V8), re.IGNORECASE)


def v8_norm(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = s.replace("\u00a0", " ")
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def v8_weird_count(s):
    s = v8_norm(s)
    return len(WEIRD_RE_V8.findall(s)) + len(NON_VIET_LATIN_RE_V8.findall(s)) * 3


def v8_accent_count(s):
    return sum(1 for ch in v8_norm(s) if ch in VIET_CHARS_V8)


def v8_digit_tokens(s):
    return re.findall(r"\d+(?:[.,]\d+)?", v8_norm(s))


def v8_similarity(a, b):
    a, b = v8_norm(a), v8_norm(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def v8_vi_score(s):
    s = v8_norm(s)
    if not s:
        return -999.0
    return (
        v8_accent_count(s) * 1.5
        - v8_weird_count(s) * 4.0
        - len(re.findall(r"[€#ñ￾]", s)) * 8.0
    )


def v8_postcorrect_text(s):
    """Conservative post-correction after OCR."""
    s = v8_norm(s)

    rules = [
        (r"\bth[eéế]\s+ky\b", "thế kỷ"),
        (r"\bth[eéế]\s+kỷ\b", "thế kỷ"),
        (r"\bdoi\s+#?ự\s+Đức\b", "đời Tự Đức"),
        (r"\bDự\s+Đức\b", "Tự Đức"),
        (r"\b#ự\s+Đức\b", "Tự Đức"),
        (r"\bMinh\s+M[ée]nh\b", "Minh Mệnh"),
        (r"\bMinh\s+Mộệnh\b", "Minh Mệnh"),
        (r"\bThiệu\s+Tri\b", "Thiệu Trị"),
        (r"\bHién\s+Tông\b", "Hiến Tông"),
        (r"\bNHẤT\s+THONG\s+CHÍ\b", "NHẤT THỐNG CHÍ"),
        (r"\bĐẠI\s+NAM\s+NHẬT\s+THỐNG\s+CHÍ\b", "ĐẠI NAM NHẤT THỐNG CHÍ"),
        (r"\bbi€n\b", "biển"),
        (r"\bki€m\b", "kiêm"),
        (r"\bmi€u\b", "miếu"),
        (r"\bchi€m\b", "chiếm"),
        (r"\bCao\s+Mén\b", "Cao Mên"),
        (r"\bHa\s+Tien\b", "Hà Tiên"),
        (r"\bQuang\s+Binh\b", "Quảng Bình"),
        (r"\bBinh\s+Dinh\b", "Bình Định"),
        (r"\bQuang\s+Yen\b", "Quảng Yên"),
        (r"\bTỈNH\s+HA\s+TIEN\b", "TỈNH HÀ TIÊN"),
        (r"\bTỈNH\s+QUANG\s+BINH\b", "TỈNH QUẢNG BÌNH"),
        (r"\bTỈNH\s+BINH\s+DINH\b", "TỈNH BÌNH ĐỊNH"),
        (r"\bTỈNH\s+QUANG\s+YEN\b", "TỈNH QUẢNG YÊN"),
        (r"(\d)\s+([,.])\s+(\d)", r"\1\2\3"),
        (r"(\d)([A-Za-zÀ-ỹ])", r"\1 \2"),
        (r"([A-Za-zÀ-ỹ])(\d)", r"\1 \2"),
    ]

    for pat, repl in rules:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)

    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def v8_find_text_col(df):
    for c in [
        "final_text_strong_ocr_v7_gpu",
        "final_text_strong_ocr_v6",
        "v5_final_text",
        "final_text_postcorrected_v4",
        "final_text",
        "auto_choose_text",
        "auto_text",
        "raw_text",
        "text",
    ]:
        if c in df.columns:
            return c
    raise ValueError("Cannot find usable text column.")


TEXT_COL_V8 = v8_find_text_col(df_v8)
print("TEXT_COL_V8:", TEXT_COL_V8)

# ---------------- Candidate selection ----------------
def v8_is_candidate(row):
    text = v8_norm(row.get(TEXT_COL_V8, ""))
    raw = v8_norm(row.get("raw_text", ""))
    auto = v8_norm(row.get("auto_text", ""))

    try:
        q = float(row.get("quality_score", 0) or 0)
    except Exception:
        q = 0.0

    if q >= V8_MIN_QUALITY_SCORE:
        return True
    if BAD_OCR_RE_V8.search(text) or BAD_OCR_RE_V8.search(raw) or BAD_OCR_RE_V8.search(auto):
        return True
    if v8_weird_count(text) >= 1:
        return True

    letters = re.findall(r"[A-Za-zÀ-ỹ]", text)
    if len(letters) >= 30:
        acc_ratio = v8_accent_count(text) / max(1, len(letters))
        if acc_ratio < 0.035 and re.search(r"\b(tinh|huyen|phu|chau|thong|nhat|kinh|song|nui|bien)\b", text, re.I):
            return True

    return False


cand_mask_v8 = df_v8.apply(v8_is_candidate, axis=1)
cand_df_v8 = df_v8[cand_mask_v8].copy()

need_cols = ["pdf_path", "page_idx", "bbox"]
missing_cols = [c for c in need_cols if c not in df_v8.columns]
if missing_cols:
    print("WARNING missing columns for re-render crop:", missing_cols)

if "bbox" in cand_df_v8.columns:
    cand_df_v8 = cand_df_v8[cand_df_v8["bbox"].fillna("").astype(str).str.len() > 0].copy()

cand_df_v8 = cand_df_v8.head(V8_MAX_ROWS).copy()
print("V8 candidates:", len(cand_df_v8), "/", len(df_v8))

# ---------------- PDF / crop helpers ----------------
def v8_parse_bbox(x):
    if isinstance(x, (list, tuple)) and len(x) >= 4:
        return [float(v) for v in x[:4]]

    s = v8_norm(x)
    if not s:
        return None

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)) and len(obj) >= 4:
            return [float(v) for v in obj[:4]]
    except Exception:
        pass

    nums = re.findall(r"-?\d+(?:\.\d+)?", s)
    if len(nums) >= 4:
        return [float(v) for v in nums[:4]]

    return None


def v8_resolve_pdf_path(pdf_path, work_id=None):
    p = Path(v8_norm(pdf_path))
    if p.exists():
        return p

    for name in ["PDF_PATHS", "pdf_paths", "pdf_files"]:
        obj = globals().get(name)
        if obj:
            try:
                if isinstance(obj, dict) and work_id in obj:
                    pp = Path(obj[work_id])
                    if pp.exists():
                        return pp
                for pp in obj:
                    pp = Path(pp)
                    if pp.name == p.name and pp.exists():
                        return pp
            except Exception:
                pass

    roots = []
    if "INPUT_ROOTS" in globals():
        roots += [Path(r) for r in INPUT_ROOTS]
    roots += [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path("/mnt/data")]

    target_names = []
    if p.name:
        target_names.append(p.name)
    if work_id:
        target_names += [f"{work_id}.pdf", f"{str(work_id).upper()}.pdf"]

    for root in roots:
        if not root.exists():
            continue
        for name in target_names:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]

    return None


def v8_make_variants(in_path, prefix_path):
    """Create multiple crop variants and return paths. Avoid hard threshold by default."""
    img = Image.open(in_path).convert("RGB")
    w, h = img.size
    if max(w, h) < 1400:
        img = img.resize((w * 2, h * 2))

    variants = []

    raw_path = prefix_path.with_name(prefix_path.stem + "_raw.png")
    img.save(raw_path)
    variants.append(raw_path)

    gray = ImageOps.grayscale(img)
    gray = ImageOps.autocontrast(gray)

    light = ImageEnhance.Contrast(gray).enhance(1.35)
    light = ImageEnhance.Sharpness(light).enhance(1.3)
    light_path = prefix_path.with_name(prefix_path.stem + "_light.png")
    light.convert("RGB").save(light_path)
    variants.append(light_path)

    strong = ImageEnhance.Contrast(gray).enhance(1.8)
    strong = ImageEnhance.Sharpness(strong).enhance(1.7)
    strong = strong.filter(ImageFilter.MedianFilter(size=3))
    strong_path = prefix_path.with_name(prefix_path.stem + "_strong.png")
    strong.convert("RGB").save(strong_path)
    variants.append(strong_path)

    return variants


def v8_render_crop_variants(row):
    sent_id = v8_norm(row.get("sent_id", row.name))
    work_id = v8_norm(row.get("work_id", "unknown"))

    if "page_idx" in row:
        page_idx = int(row.get("page_idx"))
    else:
        page_idx = int(row.get("page_number", 1)) - 1

    bbox = v8_parse_bbox(row.get("bbox", ""))
    pdf_path = v8_resolve_pdf_path(row.get("pdf_path", ""), work_id=work_id)
    if pdf_path is None or bbox is None:
        return []

    base_path = V8_CROP_DIR / f"{sent_id}_p{page_idx+1}.png"
    prefix_path = V8_CROP_DIR / f"{sent_id}_p{page_idx+1}_prep.png"

    try:
        doc = fitz.open(str(pdf_path))
        page = doc[page_idx]

        rect = fitz.Rect(*bbox)
        rect.x0 = max(0, rect.x0 - V8_PAD_PT)
        rect.y0 = max(0, rect.y0 - V8_PAD_PT)
        rect.x1 = min(page.rect.x1, rect.x1 + V8_PAD_PT)
        rect.y1 = min(page.rect.y1, rect.y1 + V8_PAD_PT)

        mat = fitz.Matrix(V8_ZOOM, V8_ZOOM)
        pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
        pix.save(str(base_path))
        doc.close()

        if V8_USE_CROP_VARIANTS:
            return v8_make_variants(base_path, prefix_path)
        return [base_path]
    except Exception as e:
        print("render_failed:", row.get("sent_id", ""), e)
        return []

# ---------------- PaddleOCR engine + parser ----------------
def v8_import_paddleocr_class_no_torch():
    """Import PaddleOCR while hiding broken Kaggle torch from ModelScope.

    PaddleOCR 3.x imports PaddleX, PaddleX imports ModelScope, and ModelScope
    may import torch only to check distributed logging. On some Kaggle images,
    importing torch fails with:
        libtorch_cuda.so: undefined symbol: ncclCommShrink
    OCR does not need torch, so we temporarily make importlib.util.find_spec("torch")
    return None during the PaddleOCR import.
    """
    import importlib.util as iutil
    import sys

    original_find_spec = iutil.find_spec

    def find_spec_without_torch(name, *args, **kwargs):
        if name == "torch" or name.startswith("torch."):
            return None
        return original_find_spec(name, *args, **kwargs)

    iutil.find_spec = find_spec_without_torch
    try:
        from paddleocr import PaddleOCR
        return PaddleOCR
    finally:
        iutil.find_spec = original_find_spec


def v8_build_paddleocr():
    """Build PaddleOCR robustly on Kaggle.

    Fixes:
    - Avoid Kaggle Torch/NCCL import crash from ModelScope.
    - Use PP-OCRv5 for Vietnamese. PP-OCRv4 + lang="vi" is not available.
    - Do not pass engine="paddle"; some PaddleOCR 3.x builds reject it.
    - Optionally use GPU if paddlepaddle-gpu is installed and V8_FORCE_GPU_IF_AVAILABLE=True.
    - Give a clear error when models cannot be downloaded because Kaggle Internet is OFF.
    """
    import os
    import urllib.request
    import paddle

    os.environ["FLAGS_use_mkldnn"] = "0"
    os.environ["FLAGS_use_onednn"] = "0"
    os.environ["FLAGS_enable_pir_api"] = "0"
    # Keep source check enabled by default so PaddleOCR can download models when Internet is ON.
    os.environ.setdefault("DISABLE_MODEL_SOURCE_CHECK", "False")

    print("paddle version:", paddle.__version__)
    print("cuda compiled:", paddle.is_compiled_with_cuda())
    try:
        print("current paddle device:", paddle.device.get_device())
    except Exception:
        pass

    if V8_FORCE_GPU_IF_AVAILABLE:
        if paddle.is_compiled_with_cuda():
            try:
                paddle.set_device("gpu:0")
                print("set paddle device:", paddle.device.get_device())
            except Exception as e:
                print("Could not set gpu:0:", repr(e))
        else:
            print("V8_FORCE_GPU_IF_AVAILABLE=True but this Paddle build is CPU-only.")

    # Import PaddleOCR after hiding torch from ModelScope.
    try:
        PaddleOCR = v8_import_paddleocr_class_no_torch()
    except Exception as e:
        raise RuntimeError(
            "Cannot import PaddleOCR. If the error mentions libtorch_cuda.so or ncclCommShrink, "
            "it is Kaggle Torch/NCCL mismatch; this notebook already hides torch during import, "
            "so restart the session and make sure no earlier cell imported torch. Original error: " + repr(e)
        )

    # Quick connectivity hint. PaddleOCR will still do its own download,
    # but this makes the Kaggle Internet=Off problem obvious.
    host_checks = [
        "https://huggingface.co",
        "https://modelscope.cn",
        "https://paddle-model-ecology.bj.bcebos.com",
    ]
    reachable = False
    for url in host_checks:
        try:
            urllib.request.urlopen(url, timeout=5)
            reachable = True
            print("model host reachable:", url)
            break
        except Exception:
            pass
    if not reachable:
        print("WARNING: No PaddleOCR model host reachable. Turn Kaggle Internet ON or provide local model cache.")

    attempts = []

    def add_attempt(**kwargs):
        # Try device argument first if requested; if unsupported, the no-device attempt below will catch it.
        if V8_FORCE_GPU_IF_AVAILABLE and paddle.is_compiled_with_cuda():
            kw = dict(kwargs)
            kw["device"] = "gpu:0"
            attempts.append(kw)
        attempts.append(kwargs)

    # Best attempt for Vietnamese in PaddleOCR 3.x.
    add_attempt(
        lang="vi",
        ocr_version="PP-OCRv5",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    # Fallback multilingual/Latin model. Different PaddleOCR builds may use one of these language ids.
    add_attempt(
        lang="latin",
        ocr_version="PP-OCRv5",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    add_attempt(
        lang="en",
        ocr_version="PP-OCRv5",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    # Last fallback: default pipeline without lang/version.
    add_attempt(
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )

    last_err = None
    for kwargs in attempts:
        try:
            print("Trying PaddleOCR init:", kwargs)
            engine = PaddleOCR(**kwargs)
            print("PaddleOCR initialized.")
            return engine
        except Exception as e:
            print("Init failed:", repr(e))
            last_err = e

    raise RuntimeError(f"Cannot initialize PaddleOCR: {last_err}")

def v8_to_plain_obj(obj):
    if obj is None:
        return None
    if isinstance(obj, (dict, list, tuple, str, int, float)):
        return obj
    for attr in ["json", "to_dict", "dict"]:
        if hasattr(obj, attr):
            try:
                val = getattr(obj, attr)
                val = val() if callable(val) else val
                return val
            except Exception:
                pass
    if hasattr(obj, "res"):
        try:
            return getattr(obj, "res")
        except Exception:
            pass
    return obj


def v8_parse_paddle_result(res):
    texts = []
    scores = []

    def add(t, sc=None):
        t = v8_norm(t)
        if not t:
            return
        if len(t) > 3000:
            return
        texts.append(t)
        try:
            scores.append(float(sc))
        except Exception:
            pass

    def parse_known(x):
        x = v8_to_plain_obj(x)
        if x is None:
            return

        if isinstance(x, dict):
            for container_key in ["res", "ocr_result", "data", "result"]:
                if container_key in x:
                    parse_known(x[container_key])

            rec_texts = x.get("rec_texts") or x.get("texts") or x.get("text") or x.get("transcription") or []
            rec_scores = x.get("rec_scores") or x.get("scores") or x.get("score") or x.get("confidence") or []

            if isinstance(rec_texts, str):
                add(rec_texts, rec_scores if isinstance(rec_scores, (int, float)) else None)
            elif isinstance(rec_texts, (list, tuple)):
                for i, t in enumerate(rec_texts):
                    sc = None
                    if isinstance(rec_scores, (list, tuple)) and i < len(rec_scores):
                        sc = rec_scores[i]
                    elif isinstance(rec_scores, (int, float)):
                        sc = rec_scores
                    add(t, sc)

            for list_key in ["lines", "items", "ocr_res", "boxes"]:
                if list_key in x and isinstance(x[list_key], (list, tuple)):
                    parse_known(x[list_key])
            return

        if isinstance(x, (list, tuple)):
            if len(x) >= 2 and isinstance(x[1], (list, tuple)):
                rec = x[1]
                if len(rec) >= 1 and isinstance(rec[0], str):
                    sc = rec[1] if len(rec) >= 2 else None
                    add(rec[0], sc)
                    return
            if len(x) >= 2 and isinstance(x[1], str):
                sc = x[2] if len(x) >= 3 else None
                add(x[1], sc)
                return
            for item in x:
                parse_known(item)
            return

    debug_type = type(res).__name__
    parse_known(res)
    text = v8_norm(" ".join(texts))
    conf = float(sum(scores) / len(scores)) if scores else 0.0
    return text, conf, debug_type


def v8_paddle_ocr_image(engine, image_path):
    image_path = str(image_path)
    if not image_path or not Path(image_path).exists():
        return "", 0.0, "missing_image"

    raw_errors = []
    if not hasattr(engine, "predict"):
        return "", 0.0, "engine_has_no_predict"

    for call_name, fn in [
        ("predict_input_kw", lambda: engine.predict(input=image_path)),
        ("predict_positional", lambda: engine.predict(image_path)),
    ]:
        try:
            res = fn()
            text, conf, dtype = v8_parse_paddle_result(res)
            if text:
                return text, conf, f"{call_name}:{dtype}"
        except Exception as e:
            raw_errors.append(f"{call_name}: {type(e).__name__}: {e}")

    return "", 0.0, "empty_or_parse_failed | " + " | ".join(raw_errors[:4])


def v8_ocr_candidate_score(base_text, ocr_text, conf):
    base = v8_norm(base_text)
    ocr = v8_norm(ocr_text)
    if not ocr:
        return -9999.0
    sim = v8_similarity(base, ocr)
    return (
        conf * 8.0
        + sim * 6.0
        + v8_accent_count(ocr) * 0.25
        - v8_weird_count(ocr) * 2.0
        - abs(len(ocr) - len(base)) / max(1, len(base)) * 3.0
    )


def v8_paddle_ocr_variants(engine, image_paths, base_text=""):
    rows = []
    for p in image_paths:
        text, conf, debug = v8_paddle_ocr_image(engine, p)
        rows.append({
            "path": str(p),
            "text": text,
            "conf": conf,
            "debug": debug,
            "variant_score": v8_ocr_candidate_score(base_text, text, conf),
        })
    if not rows:
        return "", 0.0, "no_variants", ""
    best = max(rows, key=lambda x: x["variant_score"])
    return best["text"], best["conf"], best["debug"], best["path"]

# Build engine once. Do not crash the whole notebook if PaddleOCR cannot download/init models.
PADDLEOCR_V8_AVAILABLE = False
PADDLEOCR_V8_ERROR = ""
paddle_v8 = None

if ENABLE_STRONG_OCR_V8:
    try:
        paddle_v8 = v8_build_paddleocr()
        PADDLEOCR_V8_AVAILABLE = True
    except Exception as e:
        PADDLEOCR_V8_ERROR = repr(e)
        print("\nWARNING: PaddleOCR v8 is unavailable.")
        print("Reason:", PADDLEOCR_V8_ERROR)
        print("The notebook will skip strong OCR and export base/post-corrected outputs.")
        print("Fix: turn Kaggle Internet ON for first model download, or install/provide local PaddleOCR models.")

if not PADDLEOCR_V8_AVAILABLE:
    # Avoid rendering thousands of OCR crops when OCR engine is unavailable.
    cand_df_v8 = cand_df_v8.head(0).copy()


# ---------------- Smoke test rows: skip cover pages ----------------
smoke_pool = cand_df_v8.copy()

if "page_number" in smoke_pool.columns:
    smoke_pool["_page_num_tmp"] = pd.to_numeric(smoke_pool["page_number"], errors="coerce").fillna(0)
    smoke_pool = smoke_pool[smoke_pool["_page_num_tmp"] >= 3].copy()


def v8_smoke_priority(row):
    text = v8_norm(row.get(TEXT_COL_V8, ""))
    score = 0
    if re.search(r"[€#ñ]", text):
        score += 10
    if BAD_OCR_RE_V8.search(text):
        score += 8
    if v8_weird_count(text) > 0:
        score += 5
    if re.search(r"\b(th[eéế]\s+ky|#?ự\s+Đức|bi€n|ki€m|mi€u|chi€m|ñăm|Nii)\b", text, re.I):
        score += 10
    if len(text) > 40:
        score += 2
    return score


if len(smoke_pool) == 0:
    print("WARNING: smoke_pool empty after skipping cover pages. Fallback to cand_df_v8.")
    smoke_pool = cand_df_v8.copy()

smoke_pool["_smoke_priority"] = smoke_pool.apply(v8_smoke_priority, axis=1)
smoke_rows = smoke_pool.sort_values("_smoke_priority", ascending=False).head(V8_SMOKE_TEST_ROWS).copy()
print("Smoke rows:", len(smoke_rows))

# ---------------- Run smoke test ----------------
smoke_records = []
print("\nRunning v8 PaddleOCR smoke test...")

for idx, row in smoke_rows.iterrows():
    base_text = v8_norm(row.get(TEXT_COL_V8, ""))
    crop_paths = v8_render_crop_variants(row)

    if not crop_paths:
        ocr_text, conf, debug, best_crop = "", 0.0, "render_failed", ""
    else:
        ocr_text, conf, debug, best_crop = v8_paddle_ocr_variants(paddle_v8, crop_paths, base_text=base_text)

    smoke_records.append({
        "idx": idx,
        "sent_id": row.get("sent_id", ""),
        "work_id": row.get("work_id", ""),
        "page_number": row.get("page_number", ""),
        "base_text": base_text,
        "best_crop_path": best_crop,
        "all_crop_paths": " | ".join(str(p) for p in crop_paths),
        "ocr_text": ocr_text,
        "ocr_conf": conf,
        "debug": debug,
        "base_weird": v8_weird_count(base_text),
        "ocr_weird": v8_weird_count(ocr_text),
        "base_accents": v8_accent_count(base_text),
        "ocr_accents": v8_accent_count(ocr_text),
        "similarity": v8_similarity(base_text, ocr_text),
    })

smoke_df_v8 = pd.DataFrame(smoke_records)
smoke_csv = V8_REVIEW_DIR / "strong_ocr_v8_smoke_test.csv"
smoke_df_v8.to_csv(smoke_csv, index=False, encoding="utf-8-sig")

print("Smoke test saved:", smoke_csv)
if "ocr_text" in smoke_df_v8.columns:
    non_empty = int((smoke_df_v8["ocr_text"].fillna("").astype(str).str.len() > 0).sum())
else:
    non_empty = 0
print(f"Smoke OCR non-empty: {non_empty} / {len(smoke_df_v8)}")

for _, r in smoke_df_v8.iterrows():
    print("\n---")
    print("sent_id:", r["sent_id"], "| page:", r["page_number"])
    print("base:", str(r["base_text"])[:350])
    print("ocr :", str(r["ocr_text"])[:350])
    print(
        "conf:", r["ocr_conf"],
        "| sim:", r["similarity"],
        "| accent:", f'{r["base_accents"]}->{r["ocr_accents"]}',
        "| weird:", f'{r["base_weird"]}->{r["ocr_weird"]}',
        "| debug:", r["debug"],
    )

print("\nNext:")
print("- If Smoke OCR non-empty > 0, run Cell 13B.")
print("- Cell 13B uses conservative accept and will not replace good base text blindly.")

In [ ]:
# ============================================================
# 13B. Run Strong OCR v8 + conservative accept + export final
# ============================================================

assert "paddle_v8" in globals(), "Run Cell 13A first."
assert "cand_df_v8" in globals(), "Run Cell 13A first."
assert "df_v8" in globals(), "Run Cell 13A first."

if paddle_v8 is None and len(cand_df_v8) > 0:
    print("WARNING: paddle_v8 is None, so candidates are cleared and only base/post-corrected text will be exported.")
    cand_df_v8 = cand_df_v8.head(0).copy()


def v8_digits_preserved(base, ocr):
    base_digits = v8_digit_tokens(base)
    if not base_digits:
        return True
    ocr_digits = v8_digit_tokens(ocr)
    if not ocr_digits:
        return False
    kept = sum(1 for d in base_digits if d in ocr_digits)
    return kept / max(1, len(base_digits)) >= 0.70


def v8_choose_text(base_text, ocr_text, ocr_conf):
    base = v8_postcorrect_text(base_text)
    ocr = v8_postcorrect_text(ocr_text)

    if not ocr:
        return base, "no_ocr", False

    # Reject OCR if it clearly does not correspond to the same sentence.
    # Current sentence bboxes come from paragraph bboxes, so a too-long OCR result often means paragraph bleed.
    if len(ocr) < 0.60 * max(1, len(base)):
        return base, f"reject_too_short(conf={ocr_conf:.2f},len={len(base)}->{len(ocr)})", False

    if len(ocr) > V8_MAX_OCR_LENGTH_RATIO * max(1, len(base)):
        return base, (
            f"reject_too_long_possible_paragraph_bleed(conf={ocr_conf:.2f},"
            f"len={len(base)}->{len(ocr)})"
        ), False

    if not v8_digits_preserved(base, ocr):
        return base, f"reject_lost_digits(conf={ocr_conf:.2f})", False

    sim = v8_similarity(base, ocr)
    base_weird = v8_weird_count(base)
    ocr_weird = v8_weird_count(ocr)
    base_acc = v8_accent_count(base)
    ocr_acc = v8_accent_count(ocr)
    base_bad = bool(BAD_OCR_RE_V8.search(base))
    ocr_bad = bool(BAD_OCR_RE_V8.search(ocr))
    base_score = v8_vi_score(base)
    ocr_score = v8_vi_score(ocr)

    if NON_VIET_LATIN_RE_V8.search(ocr):
        return base, f"reject_non_viet_latin(conf={ocr_conf:.2f})", False

    # Similarity is the strongest guard against high-confidence OCR from the wrong crop.
    if sim < V8_ACCEPT_MIN_SIM:
        maybe_fix_bad_base = (
            base_bad
            and not ocr_bad
            and ocr_conf >= 0.70
            and sim >= V8_ACCEPT_BAD_MIN_SIM
            and ocr_weird < base_weird
            and ocr_score >= base_score + 8.0
        )
        if not maybe_fix_bad_base:
            return base, (
                f"reject_low_similarity_possible_wrong_crop(conf={ocr_conf:.2f},sim={sim:.2f},"
                f"len={len(base)}->{len(ocr)},score={base_score:.1f}->{ocr_score:.1f})"
            ), False

    # If base already has many Vietnamese accents, do not replace with accent-poor OCR.
    if base_acc >= 4 and ocr_acc < base_acc * 0.65:
        return base, (
            f"reject_accent_loss(conf={ocr_conf:.2f},sim={sim:.2f},"
            f"accent={base_acc}->{ocr_acc})"
        ), False

    if ocr_weird > base_weird:
        return base, (
            f"reject_more_weird(conf={ocr_conf:.2f},sim={sim:.2f},"
            f"weird={base_weird}->{ocr_weird})"
        ), False

    # Accept only when base has a known bad pattern and OCR removes it cleanly.
    if (
        base_bad
        and not ocr_bad
        and ocr_conf >= 0.70
        and sim >= V8_ACCEPT_BAD_MIN_SIM
        and ocr_weird <= base_weird
        and len(ocr) >= 0.80 * max(1, len(base))
        and len(ocr) <= V8_MAX_OCR_LENGTH_RATIO * max(1, len(base))
        and ocr_score >= base_score + 8.0
    ):
        return ocr, (
            f"accept_fix_bad_pattern(conf={ocr_conf:.2f},sim={sim:.2f},"
            f"weird={base_weird}->{ocr_weird},accent={base_acc}->{ocr_acc},"
            f"score={base_score:.1f}->{ocr_score:.1f})"
        ), True

    # Accept only when OCR is clearly better as Vietnamese text.
    if (
        ocr_conf >= 0.80
        and sim >= V8_ACCEPT_MIN_SIM
        and ocr_score >= base_score + 8.0
        and ocr_acc >= base_acc
        and ocr_weird <= base_weird
    ):
        return ocr, (
            f"accept_better_vi_score(conf={ocr_conf:.2f},sim={sim:.2f},"
            f"accent={base_acc}->{ocr_acc},score={base_score:.1f}->{ocr_score:.1f})"
        ), True

    return base, (
        f"keep_base(conf={ocr_conf:.2f},sim={sim:.2f},"
        f"weird={base_weird}->{ocr_weird},accent={base_acc}->{ocr_acc},"
        f"score={base_score:.1f}->{ocr_score:.1f},base_bad={base_bad},ocr_bad={ocr_bad})"
    ), False


run_df_v8 = df_v8.copy()
run_df_v8["base_text_v8"] = run_df_v8[TEXT_COL_V8].fillna("").astype(str).map(v8_norm)
run_df_v8["strong_ocr_v8_text"] = ""
run_df_v8["strong_ocr_v8_conf"] = 0.0
run_df_v8["strong_ocr_v8_debug"] = ""
run_df_v8["strong_ocr_v8_crop"] = ""
run_df_v8["strong_ocr_v8_reason"] = "not_selected"
run_df_v8["strong_ocr_v8_accepted"] = False
run_df_v8["final_text_strong_ocr_v8"] = run_df_v8["base_text_v8"].map(v8_postcorrect_text)
run_df_v8["final_source_strong_ocr_v8"] = "base_postcorrected"

records = []
raw_debug_records = []

print("Running Strong OCR v8 candidates:", len(cand_df_v8))
if globals().get("PADDLEOCR_V8_ERROR"):
    print("PaddleOCR skipped due to:", globals().get("PADDLEOCR_V8_ERROR"))


for n, (idx, row) in enumerate(tqdm(cand_df_v8.iterrows(), total=len(cand_df_v8), desc="strong OCR v8"), 1):
    base_text = v8_norm(run_df_v8.at[idx, "base_text_v8"])

    crop_paths = v8_render_crop_variants(row)
    if not crop_paths:
        ocr_text, ocr_conf, debug, best_crop = "", 0.0, "render_failed", ""
    else:
        ocr_text, ocr_conf, debug, best_crop = v8_paddle_ocr_variants(paddle_v8, crop_paths, base_text=base_text)

    chosen, reason, accepted = v8_choose_text(base_text, ocr_text, ocr_conf)

    if not V8_AUTO_ACCEPT:
        chosen = v8_postcorrect_text(base_text)
        accepted = False
        reason = "auto_accept_disabled | " + reason

    run_df_v8.at[idx, "strong_ocr_v8_text"] = ocr_text
    run_df_v8.at[idx, "strong_ocr_v8_conf"] = ocr_conf
    run_df_v8.at[idx, "strong_ocr_v8_debug"] = debug
    run_df_v8.at[idx, "strong_ocr_v8_crop"] = best_crop
    run_df_v8.at[idx, "strong_ocr_v8_reason"] = reason
    run_df_v8.at[idx, "strong_ocr_v8_accepted"] = bool(accepted)
    run_df_v8.at[idx, "final_text_strong_ocr_v8"] = chosen
    run_df_v8.at[idx, "final_source_strong_ocr_v8"] = "strong_ocr_v8" if accepted else "base_postcorrected"

    records.append({
        "idx": idx,
        "sent_id": row.get("sent_id", ""),
        "work_id": row.get("work_id", ""),
        "page_number": row.get("page_number", ""),
        "base_text": base_text,
        "ocr_text": ocr_text,
        "ocr_conf": ocr_conf,
        "accepted": bool(accepted),
        "reason": reason,
        "best_crop_path": best_crop,
        "debug": debug,
    })

    if (not ocr_text) and len(raw_debug_records) < V8_DEBUG_RAW_LIMIT:
        raw_debug_records.append({
            "idx": idx,
            "sent_id": row.get("sent_id", ""),
            "page_number": row.get("page_number", ""),
            "best_crop_path": best_crop,
            "debug": debug,
            "base_text": base_text,
        })


audit_v8 = pd.DataFrame(records)


def v8_remaining_issue_reason(text):
    text = v8_norm(text)
    reasons = []
    if BAD_OCR_RE_V8.search(text):
        reasons.append("bad_pattern")
    if v8_weird_count(text) >= 1:
        reasons.append("weird_char")
    if re.search(r"[A-Za-zÀ-ỹ]\d|\d[A-Za-zÀ-ỹ]", text):
        reasons.append("digit_letter_glue")
    if len(text) >= 40:
        letters = re.findall(r"[A-Za-zÀ-ỹ]", text)
        if letters:
            acc_ratio = v8_accent_count(text) / max(1, len(letters))
            if acc_ratio < 0.025 and re.search(r"\b(tinh|huyen|phu|chau|thong|nhat|kinh|song|nui|bien)\b", text, re.I):
                reasons.append("low_accent_historical_text")
    return ";".join(reasons)


run_df_v8["remaining_issue_reason_v8"] = run_df_v8["final_text_strong_ocr_v8"].map(v8_remaining_issue_reason)
remaining_v8 = run_df_v8[run_df_v8["remaining_issue_reason_v8"].fillna("").astype(str).str.len() > 0].copy()

# Save CSV outputs.
final_csv_v8 = V8_FINAL_DIR / "final_sentences_strong_ocr_v8.csv"
audit_csv_v8 = V8_REVIEW_DIR / "strong_ocr_v8_audit.csv"
remaining_csv_v8 = V8_REVIEW_DIR / "remaining_ocr_issues_after_v8.csv"
debug_csv_v8 = V8_REVIEW_DIR / "strong_ocr_v8_empty_debug.csv"

run_df_v8.to_csv(final_csv_v8, index=False, encoding="utf-8-sig")
audit_v8.to_csv(audit_csv_v8, index=False, encoding="utf-8-sig")
remaining_v8.to_csv(remaining_csv_v8, index=False, encoding="utf-8-sig")
pd.DataFrame(raw_debug_records).to_csv(debug_csv_v8, index=False, encoding="utf-8-sig")

print("Saved final:", final_csv_v8)
print("Saved audit:", audit_csv_v8)
print("Saved remaining:", remaining_csv_v8)
print("Saved debug:", debug_csv_v8)

ocr_non_empty = int((audit_v8["ocr_text"].fillna("").astype(str).str.len() > 0).sum()) if len(audit_v8) else 0
accepted_n = int(audit_v8["accepted"].sum()) if len(audit_v8) else 0

print("\nStrong OCR v8 summary")
print("Candidates:", len(cand_df_v8))
print("OCR non-empty:", ocr_non_empty)
print("Accepted:", accepted_n)
print("Remaining flagged rows:", len(remaining_v8))

# Export text files by work_id.
for work_id, g in run_df_v8.groupby("work_id", dropna=False):
    work_id = v8_norm(work_id) or "unknown"
    out_txt = V8_TEXT_DIR / f"{work_id}_final_strong_ocr_v8.txt"

    parts = []
    if "page_number" in g.columns:
        sort_cols = ["page_number"]
        if "sent_id" in g.columns:
            sort_cols.append("sent_id")
        g = g.sort_values(sort_cols)

    current_page = None
    for _, r in g.iterrows():
        page = r.get("page_number", "")
        if page != current_page:
            current_page = page
            parts.append(f"\n\n[Page {page}]\n")
        txt = v8_norm(r.get("final_text_strong_ocr_v8", ""))
        if txt:
            parts.append(txt)

    out_txt.write_text("\n".join(parts), encoding="utf-8")

print("Text files saved to:", V8_TEXT_DIR)

summary_v8 = pd.DataFrame([{
    "input_csv": str(INPUT_SENT_CSV),
    "rows_total": len(run_df_v8),
    "candidates": len(cand_df_v8),
    "ocr_non_empty": ocr_non_empty,
    "accepted": accepted_n,
    "remaining_flagged_rows": len(remaining_v8),
}])
summary_csv_v8 = V8_REVIEW_DIR / "strong_ocr_v8_summary.csv"
summary_v8.to_csv(summary_csv_v8, index=False, encoding="utf-8-sig")
print("Summary:", summary_csv_v8)

zip_path_v8 = V8_PACKAGE_DIR / "dntc_full_pipeline_strong_ocr_v8.zip"
if zip_path_v8.exists():
    zip_path_v8.unlink()

with zipfile.ZipFile(zip_path_v8, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [final_csv_v8, audit_csv_v8, remaining_csv_v8, debug_csv_v8, summary_csv_v8]:
        if Path(p).exists():
            zf.write(p, arcname=str(Path(p).relative_to(V8_DIR)))

    for p in V8_TEXT_DIR.glob("*.txt"):
        zf.write(p, arcname=str(p.relative_to(V8_DIR)))

print("ZIP:", zip_path_v8)

In [ ]:
# ============================================================
# 13C. Enhanced line-window OCR v9
# ============================================================
# Goal: avoid paragraph-bleed OCR by re-rendering the smallest PDF line window
# that matches each suspicious sentence, then applying the same conservative guards.

assert "paddle_v8" in globals(), "Run Cell 13A first."
assert "df_v8" in globals(), "Run Cell 13A first."
assert "v8_choose_text" in globals(), "Run Cell 13B first so guards are defined."

V9_ENABLE_LINE_WINDOW_OCR = True
V9_AUTO_ACCEPT = True
V9_MAX_ROWS = 8000
V9_MAX_LINES_PER_WINDOW = 4
V9_MIN_LINE_MATCH_SIM = 0.34
V9_ZOOM = 5.5
V9_PAD_PT = 7.0
V9_USE_CROP_VARIANTS = True

V9_DIR = BASE_OUTPUT_DIR / "final_line_window_ocr_v9"
V9_FINAL_DIR = V9_DIR / "final"
V9_TEXT_DIR = V9_FINAL_DIR / "texts"
V9_REVIEW_DIR = V9_DIR / "review"
V9_CROP_DIR = V9_DIR / "line_window_crops"
V9_PACKAGE_DIR = V9_DIR / "packages"
for d in [V9_FINAL_DIR, V9_TEXT_DIR, V9_REVIEW_DIR, V9_CROP_DIR, V9_PACKAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("V9_DIR:", V9_DIR)


def v9_union_bbox(boxes):
    xs0, ys0, xs1, ys1 = zip(*boxes)
    return [min(xs0), min(ys0), max(xs1), max(ys1)]


def v9_line_from_spans(line):
    texts = []
    boxes = []
    for sp in line.get("spans", []):
        txt = sp.get("text", "")
        if txt:
            texts.append(txt)
            boxes.append(sp.get("bbox", line.get("bbox", [0, 0, 0, 0])))
    text = v8_norm("".join(texts))
    if not text:
        return None
    return {"text": text, "bbox": v9_union_bbox(boxes) if boxes else list(line.get("bbox", [0, 0, 0, 0]))}


_V9_PAGE_LINE_CACHE = {}


def v9_extract_page_lines(pdf_path, page_idx):
    key = (str(pdf_path), int(page_idx))
    if key in _V9_PAGE_LINE_CACHE:
        return _V9_PAGE_LINE_CACHE[key]

    doc = fitz.open(str(pdf_path))
    page = doc[int(page_idx)]
    data = page.get_text("dict")
    lines = []
    for block in data.get("blocks", []):
        if block.get("type", 0) != 0:
            continue
        for line in block.get("lines", []):
            item = v9_line_from_spans(line)
            if not item:
                continue
            txt = item["text"]
            if re.fullmatch(r"[-–—]?\s*\d{1,4}\s*[-–—]?", txt):
                continue
            if ".pdf" in txt.lower():
                continue
            lines.append(item)
    doc.close()
    lines.sort(key=lambda x: (round(x["bbox"][1], 1), round(x["bbox"][0], 1)))
    _V9_PAGE_LINE_CACHE[key] = lines
    return lines


def v9_compact_for_match(s):
    s = v8_norm(s).lower()
    s = re.sub(r"[^0-9a-zà-ỹăâđêôơư]+", "", s)
    return s


def v9_match_line_window(base_text, lines, max_lines=V9_MAX_LINES_PER_WINDOW):
    base = v8_norm(base_text)
    if not base or not lines:
        return [], 0.0, "empty"

    base_compact = v9_compact_for_match(base)
    best = ([], 0.0, "")

    for i in range(len(lines)):
        chunk_text = ""
        chunk_boxes = []
        for j in range(i, min(len(lines), i + max_lines)):
            chunk_text = v8_norm((chunk_text + " " + lines[j]["text"]).strip())
            chunk_boxes.append(lines[j]["bbox"])
            chunk_compact = v9_compact_for_match(chunk_text)

            sim = v8_similarity(base, chunk_text)
            if base_compact and chunk_compact:
                if base_compact in chunk_compact or chunk_compact in base_compact:
                    sim = max(sim, 0.92)

            # Penalize very wide windows; they are more likely to include neighbor sentences.
            len_ratio = len(chunk_text) / max(1, len(base))
            if len_ratio > 1.6:
                sim -= min(0.25, (len_ratio - 1.6) * 0.15)

            if sim > best[1]:
                best = (chunk_boxes[:], sim, chunk_text)

    return best


def v9_render_line_window(row, base_text):
    work_id = v8_norm(row.get("work_id", "unknown"))
    sent_id = v8_norm(row.get("sent_id", row.name))
    page_idx = int(row.get("page_idx", int(row.get("page_number", 1)) - 1))
    pdf_path = v8_resolve_pdf_path(row.get("pdf_path", ""), work_id=work_id)
    if pdf_path is None:
        return [], "missing_pdf", 0.0, ""

    lines = v9_extract_page_lines(pdf_path, page_idx)
    boxes, match_sim, matched_text = v9_match_line_window(base_text, lines)
    if not boxes or match_sim < V9_MIN_LINE_MATCH_SIM:
        return [], f"line_match_failed(sim={match_sim:.2f})", match_sim, matched_text

    bbox = v9_union_bbox(boxes)
    base_path = V9_CROP_DIR / work_id / f"{sent_id}_p{page_idx+1}_line.png"
    prefix_path = V9_CROP_DIR / work_id / f"{sent_id}_p{page_idx+1}_line_prep.png"
    base_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        doc = fitz.open(str(pdf_path))
        page = doc[page_idx]
        rect = fitz.Rect(*bbox)
        rect.x0 = max(0, rect.x0 - V9_PAD_PT)
        rect.y0 = max(0, rect.y0 - V9_PAD_PT)
        rect.x1 = min(page.rect.x1, rect.x1 + V9_PAD_PT)
        rect.y1 = min(page.rect.y1, rect.y1 + V9_PAD_PT)
        pix = page.get_pixmap(matrix=fitz.Matrix(V9_ZOOM, V9_ZOOM), clip=rect, alpha=False)
        pix.save(str(base_path))
        doc.close()
    except Exception as e:
        return [], f"render_failed:{type(e).__name__}:{e}", match_sim, matched_text

    if V9_USE_CROP_VARIANTS:
        return v8_make_variants(base_path, prefix_path), "ok", match_sim, matched_text
    return [base_path], "ok", match_sim, matched_text


# Start from the best base text available. If Cell 13B was run, reuse its run_df_v8; otherwise use df_v8.
if "run_df_v8" in globals():
    run_df_v9 = run_df_v8.copy()
else:
    run_df_v9 = df_v8.copy()
    run_df_v9["base_text_v8"] = run_df_v9[TEXT_COL_V8].fillna("").astype(str).map(v8_norm)
    run_df_v9["final_text_strong_ocr_v8"] = run_df_v9["base_text_v8"].map(v8_postcorrect_text)

run_df_v9["line_ocr_v9_text"] = ""
run_df_v9["line_ocr_v9_conf"] = 0.0
run_df_v9["line_ocr_v9_debug"] = ""
run_df_v9["line_ocr_v9_crop"] = ""
run_df_v9["line_ocr_v9_match_sim"] = 0.0
run_df_v9["line_ocr_v9_matched_text"] = ""
run_df_v9["line_ocr_v9_reason"] = "not_selected"
run_df_v9["line_ocr_v9_accepted"] = False
run_df_v9["final_text_line_ocr_v9"] = run_df_v9["final_text_strong_ocr_v8"].fillna("").astype(str).map(v8_postcorrect_text)
run_df_v9["final_source_line_ocr_v9"] = "base_or_v8_postcorrected"

if "cand_df_v8" in globals():
    cand_df_v9 = cand_df_v8.copy()
else:
    cand_df_v9 = run_df_v9[run_df_v9.apply(v8_is_candidate, axis=1)].copy()

cand_df_v9 = cand_df_v9.head(V9_MAX_ROWS).copy()
print("V9 line-window candidates:", len(cand_df_v9), "/", len(run_df_v9))

records_v9 = []

if paddle_v8 is None:
    print("WARNING: paddle_v8 is None. V9 will export base/postcorrected text only.")
    cand_df_v9 = cand_df_v9.head(0).copy()

for idx, row in tqdm(cand_df_v9.iterrows(), total=len(cand_df_v9), desc="line-window OCR v9"):
    base_text = v8_norm(run_df_v9.at[idx, "final_text_line_ocr_v9"])
    crop_paths, render_debug, line_match_sim, matched_text = v9_render_line_window(row, base_text)

    if not crop_paths:
        ocr_text, ocr_conf, ocr_debug, best_crop = "", 0.0, render_debug, ""
    else:
        ocr_text, ocr_conf, ocr_debug, best_crop = v8_paddle_ocr_variants(paddle_v8, crop_paths, base_text=base_text)

    chosen, reason, accepted = v8_choose_text(base_text, ocr_text, ocr_conf)
    if not V9_AUTO_ACCEPT:
        chosen = v8_postcorrect_text(base_text)
        accepted = False
        reason = "v9_auto_accept_disabled | " + reason

    run_df_v9.at[idx, "line_ocr_v9_text"] = ocr_text
    run_df_v9.at[idx, "line_ocr_v9_conf"] = ocr_conf
    run_df_v9.at[idx, "line_ocr_v9_debug"] = ocr_debug
    run_df_v9.at[idx, "line_ocr_v9_crop"] = best_crop
    run_df_v9.at[idx, "line_ocr_v9_match_sim"] = line_match_sim
    run_df_v9.at[idx, "line_ocr_v9_matched_text"] = matched_text
    run_df_v9.at[idx, "line_ocr_v9_reason"] = reason
    run_df_v9.at[idx, "line_ocr_v9_accepted"] = bool(accepted)
    run_df_v9.at[idx, "final_text_line_ocr_v9"] = chosen
    run_df_v9.at[idx, "final_source_line_ocr_v9"] = "line_ocr_v9" if accepted else "base_or_v8_postcorrected"

    records_v9.append({
        "idx": idx,
        "sent_id": row.get("sent_id", ""),
        "work_id": row.get("work_id", ""),
        "page_number": row.get("page_number", ""),
        "base_text": base_text,
        "matched_text": matched_text,
        "line_match_sim": line_match_sim,
        "ocr_text": ocr_text,
        "ocr_conf": ocr_conf,
        "accepted": bool(accepted),
        "reason": reason,
        "best_crop_path": best_crop,
        "debug": ocr_debug,
    })

audit_v9 = pd.DataFrame(records_v9)
run_df_v9["remaining_issue_reason_v9"] = run_df_v9["final_text_line_ocr_v9"].map(v8_remaining_issue_reason)
remaining_v9 = run_df_v9[run_df_v9["remaining_issue_reason_v9"].fillna("").astype(str).str.len() > 0].copy()

final_csv_v9 = V9_FINAL_DIR / "final_sentences_line_ocr_v9.csv"
audit_csv_v9 = V9_REVIEW_DIR / "line_ocr_v9_audit.csv"
remaining_csv_v9 = V9_REVIEW_DIR / "remaining_ocr_issues_after_v9.csv"
summary_csv_v9 = V9_REVIEW_DIR / "line_ocr_v9_summary.csv"

run_df_v9.to_csv(final_csv_v9, index=False, encoding="utf-8-sig")
audit_v9.to_csv(audit_csv_v9, index=False, encoding="utf-8-sig")
remaining_v9.to_csv(remaining_csv_v9, index=False, encoding="utf-8-sig")

ocr_non_empty_v9 = int((audit_v9["ocr_text"].fillna("").astype(str).str.len() > 0).sum()) if len(audit_v9) else 0
accepted_n_v9 = int(audit_v9["accepted"].sum()) if len(audit_v9) else 0
summary_v9 = pd.DataFrame([{
    "rows_total": len(run_df_v9),
    "candidates": len(cand_df_v9),
    "ocr_non_empty": ocr_non_empty_v9,
    "accepted": accepted_n_v9,
    "remaining_flagged_rows": len(remaining_v9),
    "min_line_match_sim": V9_MIN_LINE_MATCH_SIM,
    "auto_accept": V9_AUTO_ACCEPT,
}])
summary_v9.to_csv(summary_csv_v9, index=False, encoding="utf-8-sig")

for work_id, g in run_df_v9.groupby("work_id", dropna=False):
    work_id = v8_norm(work_id) or "unknown"
    out_txt = V9_TEXT_DIR / f"{work_id}_final_line_ocr_v9.txt"
    parts = []
    if "page_number" in g.columns:
        sort_cols = ["page_number"]
        if "sent_id" in g.columns:
            sort_cols.append("sent_id")
        g = g.sort_values(sort_cols)
    current_page = None
    for _, r in g.iterrows():
        page = r.get("page_number", "")
        if page != current_page:
            current_page = page
            parts.append(f"\n\n[Page {page}]\n")
        txt = v8_norm(r.get("final_text_line_ocr_v9", ""))
        if txt:
            parts.append(txt)
    out_txt.write_text("\n".join(parts), encoding="utf-8")

zip_path_v9 = V9_PACKAGE_DIR / "dntc_full_pipeline_line_ocr_v9.zip"
if zip_path_v9.exists():
    zip_path_v9.unlink()
with zipfile.ZipFile(zip_path_v9, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [final_csv_v9, audit_csv_v9, remaining_csv_v9, summary_csv_v9]:
        if Path(p).exists():
            zf.write(p, arcname=str(Path(p).relative_to(V9_DIR)))
    for p in V9_TEXT_DIR.glob("*.txt"):
        zf.write(p, arcname=str(p.relative_to(V9_DIR)))

print("Saved final:", final_csv_v9)
print("Saved audit:", audit_csv_v9)
print("Saved remaining:", remaining_csv_v9)
print("Summary:", summary_csv_v9)
print("Text files saved to:", V9_TEXT_DIR)
print("ZIP:", zip_path_v9)
print("V9 OCR non-empty:", ocr_non_empty_v9, "/", len(audit_v9))
print("V9 accepted:", accepted_n_v9)
print("V9 remaining flagged rows:", len(remaining_v9))